# Protected 25-Example Full-System Evaluation

This notebook creates and runs a frozen, previously unused holdout batch:

- five examples from each of E2E, WebNLG, DART, ToTTo, and SportSett;
- one Full-System generation per example;
- DeepSeek V4 Flash for all six workflow roles;
- exact task contracts from the prepared benchmark records;
- held-out references physically replaced by a sentinel during generation;
- per-example checkpoints, heartbeat logging, and resumable execution;
- exact source/config/code hashes and complete workflow-artifact indexes;
- sentence support, evidence/fact/insight, retry, audit, and token summaries.

The selection is written once and never silently recomputed. The notebook
also verifies the frozen implementation fingerprint before every generation.
Reference-based scoring is a separate, disabled-by-default final phase that
cannot run until all 25 generations have completed successfully.



In [1]:
import ast
import asyncio
import copy
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from table2text.config import Settings
from table2text.evaluation import (
    default_paths,
    generate_reports_for_notebook,
    load_project_env,
    score_reference_metrics_for_notebook,
)
from table2text.evaluation.datasets import read_examples
from table2text.evaluation.generation import (
    materialise_input,
    read_generations,
)


# =========================
# User configuration
# =========================

PROJECT_DIR = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
EXPERIMENT_ID = "protected_holdout_full_system_flash_25"

DATASETS = [
    "e2e_nlg",
    "web_nlg",
    "dart",
    "totto",
    "sportsett_basketball",
]
EXAMPLES_PER_DATASET = 5
SELECTION_SEED = 20260820
GENERATION_SEED = 42
MODEL = "deepseek:deepseek-v4-flash"

# Leave as None to run all unfinished examples. Set to 5, for example, to
# process one interleaved block and resume later without changing selection.
MAX_CASES_THIS_SESSION = None

HEARTBEAT_SECONDS = 20
MAX_TOP_LEVEL_ATTEMPTS_PER_EXAMPLE = 2
CONTINUE_AFTER_FAILURE = True
PRINT_FINAL_OUTPUTS = True

# These declarations are copied into the protected-set manifest. Keep them
# True only if they are factually correct at the moment this notebook is run.
RESEARCHER_CONFIRMS_NO_PRIOR_MANUAL_INSPECTION = True
RESEARCHER_CONFIRMS_NO_DEVELOPMENT_CHANGES_AFTER_SELECTION = True

# References are unsealed only after all 25 generations succeed. These metric
# phases are deliberately opt-in and do not alter any workflow output.
RUN_POST_GENERATION_REFERENCE_METRICS = False
RUN_POST_GENERATION_SOURCE_METRICS = False


# =========================
# Paths
# =========================

PATHS = default_paths(PROJECT_DIR)
ARTIFACT_DIR = PROJECT_DIR / "evaluation" / "protected_holdout_full_system"
CONFIG_DIR = ARTIFACT_DIR / "config"
PREPARED_DIR = ARTIFACT_DIR / "prepared"
GENERATION_DIR = ARTIFACT_DIR / "generations"
SHARD_DIR = GENERATION_DIR / "shards"
ATTEMPT_RECORD_DIR = GENERATION_DIR / "attempt_records"
RUN_ROOT = GENERATION_DIR / "runs"
RESULT_DIR = ARTIFACT_DIR / "results"
SNAPSHOT_DIR = ARTIFACT_DIR / "frozen_code_snapshot"
MODEL_INPUT_AUDIT_DIR = ARTIFACT_DIR / "model_input_audit"

for directory in (
    CONFIG_DIR,
    PREPARED_DIR,
    GENERATION_DIR,
    SHARD_DIR,
    ATTEMPT_RECORD_DIR,
    RUN_ROOT,
    RESULT_DIR,
    SNAPSHOT_DIR,
    MODEL_INPUT_AUDIT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

VARIANT_PATH = CONFIG_DIR / "protected_full_system_flash.json"
FREEZE_MANIFEST_PATH = SNAPSHOT_DIR / "freeze_manifest.json"
SELECTION_MANIFEST_PATH = PREPARED_DIR / "protected_selection_manifest.json"
OPERATIONAL_EXAMPLES_PATH = PREPARED_DIR / "protected_operational_examples.jsonl"
EXCLUSION_MANIFEST_PATH = PREPARED_DIR / "historical_exclusions.json"
BATCH_MANIFEST_PATH = RESULT_DIR / "protected_batch_manifest.json"
PROGRESS_LOG_PATH = RESULT_DIR / "protected_progress.log"
ATTEMPT_LOG_PATH = RESULT_DIR / "generation_attempts.jsonl"
SEALED_GENERATIONS_PATH = GENERATION_DIR / "protected_full_system_generations_sealed.jsonl"
UNSEALED_GENERATIONS_PATH = GENERATION_DIR / "protected_full_system_generations_post_generation.jsonl"
SUMMARY_JSONL_PATH = RESULT_DIR / "protected_generation_summary.jsonl"
SUMMARY_CSV_PATH = RESULT_DIR / "protected_generation_summary.csv"
STAGE_USAGE_CSV_PATH = RESULT_DIR / "stage_token_usage.csv"
SUPPORT_MAP_JSONL_PATH = RESULT_DIR / "sentence_support_mappings.jsonl"
ARTIFACT_INDEX_PATH = RESULT_DIR / "run_artifact_indexes.jsonl"
EXECUTION_REPORT_PATH = RESULT_DIR / "PROTECTED_HOLDOUT_EXECUTION_REPORT.md"




## Helpers

Progress messages are written both to the cell output and to a persistent
log. Atomic JSON writes prevent an interrupted kernel from leaving a partial
manifest. Long generations emit a heartbeat every 20 seconds.



In [2]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def log(message):
    line = f"[{datetime.now().strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with PROGRESS_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, "value"):
        return value.value
    raise TypeError(f"Cannot serialize {type(value).__name__}")


def write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            default=json_default,
        )
        + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def write_jsonl_atomic(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for record in records:
            if hasattr(record, "model_dump"):
                record = record.model_dump(mode="json")
            handle.write(
                json.dumps(record, ensure_ascii=False, default=json_default)
                + "\n"
            )
    temporary.replace(path)


def append_jsonl(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(record, ensure_ascii=False, default=json_default) + "\n"
        )


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl_objects(path):
    path = Path(path)
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())


def canonical_sha256(value):
    payload = json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        default=json_default,
    )
    return sha256_bytes(payload.encode("utf-8"))


def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def relative_path(path):
    path = Path(path).resolve()
    try:
        return str(path.relative_to(PROJECT_DIR.resolve()))
    except ValueError:
        return str(path)


def git_output(*arguments):
    completed = subprocess.run(
        ["git", *arguments],
        cwd=PROJECT_DIR,
        check=False,
        text=True,
        capture_output=True,
    )
    return completed.stdout.strip() if completed.returncode == 0 else None


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


async def with_heartbeat(awaitable, label, interval=HEARTBEAT_SECONDS):
    started = time.perf_counter()
    task = asyncio.create_task(awaitable)
    while True:
        try:
            result = await asyncio.wait_for(asyncio.shield(task), timeout=interval)
            elapsed = time.perf_counter() - started
            log(f"{label}: completed after {elapsed:.1f}s")
            return result
        except asyncio.TimeoutError:
            elapsed = time.perf_counter() - started
            log(f"{label}: still running ({elapsed:.1f}s elapsed)")


def markdown_word_count(text):
    without_markup = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    without_markup = re.sub(r"^#{1,6}\s+", "", without_markup, flags=re.MULTILINE)
    return len(re.findall(r"\b[\w'-]+\b", without_markup, flags=re.UNICODE))


def prose_sentence_count(text):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip() and not line.lstrip().startswith(("#", "<!--"))
    ]
    prose = " ".join(lines)
    return len(
        [part for part in re.split(r"(?<=[.!?])\s+", prose) if part.strip()]
    )


def nested_list(payload, *path):
    current = payload
    for key in path:
        if not isinstance(current, dict):
            return []
        current = current.get(key)
    return current if isinstance(current, list) else []


load_project_env(PROJECT_DIR)
log(f"Project: {PROJECT_DIR}")
log(f"Experiment: {EXPERIMENT_ID}")




[01:28:22] Project: /Users/realgobs/Documents/MScproject/table2text_pydanticai
[01:28:22] Experiment: protected_holdout_full_system_flash_25


## 1. Freeze the Full-System configuration and implementation

All environment-derived workflow settings are resolved once. The six model
roles are then fixed to DeepSeek V4 Flash, and the complete settings payload
is stored in the protected variant. A source-tree snapshot is used even if
the Git worktree is dirty; the commit and dirty status are both disclosed.
Every later run must match this fingerprint exactly.



In [3]:
MODEL_FIELDS = [
    "data_understanding_model",
    "orchestrator_model",
    "evidence_model",
    "verifier_model",
    "writer_model",
    "auditor_model",
]


def build_frozen_variant():
    resolved = asdict(Settings.from_env())
    resolved["use_llm"] = True
    for field_name in MODEL_FIELDS:
        resolved[field_name] = MODEL

    # These two are supplied per generation by the evaluation runner.
    resolved.pop("output_dir", None)
    resolved.pop("random_seed", None)

    return {
        "variants": [
            {
                "variant_id": "full_system",
                "enabled": True,
                "backend": "table2text",
                "description": (
                    "Frozen protected-holdout Full-System run with all six "
                    "roles on DeepSeek V4 Flash."
                ),
                "settings_overrides": resolved,
                "task_contract_mode": "explicit",
                "request_override": None,
                "callable_path": None,
                "command": [],
                "precomputed_path": None,
                "repetitions": 1,
                "seeds": [GENERATION_SEED],
            }
        ]
    }


if not VARIANT_PATH.exists():
    write_json_atomic(VARIANT_PATH, build_frozen_variant())
    log(f"Created frozen variant: {VARIANT_PATH}")
else:
    log(f"Reusing existing frozen variant: {VARIANT_PATH}")

frozen_variant_payload = read_json(VARIANT_PATH)
frozen_variant = frozen_variant_payload["variants"][0]
frozen_overrides = frozen_variant["settings_overrides"]

assert frozen_variant["variant_id"] == "full_system"
assert frozen_variant["task_contract_mode"] == "explicit"
assert frozen_variant["repetitions"] == 1
assert frozen_variant["seeds"] == [GENERATION_SEED]
assert all(frozen_overrides[field] == MODEL for field in MODEL_FIELDS)
assert frozen_overrides["use_llm"] is True


def implementation_files():
    files = sorted((PROJECT_DIR / "src" / "table2text").rglob("*.py"))
    files.extend(
        path
        for path in [
            PROJECT_DIR / "pyproject.toml",
            PROJECT_DIR / "evaluation" / "config" / "datasets.json",
            PROJECT_DIR
            / "evaluation"
            / "notebooks"
            / "protected_holdout_full_system_evaluation.py",
        ]
        if path.exists()
    )
    return sorted(set(path.resolve() for path in files))


def file_manifest(files):
    return [
        {
            "path": relative_path(path),
            "sha256": sha256_file(path),
            "bytes": path.stat().st_size,
        }
        for path in files
    ]


def implementation_fingerprint(files=None):
    manifest = file_manifest(files or implementation_files())
    return canonical_sha256(manifest), manifest


class AgentParameterVisitor(ast.NodeVisitor):
    def __init__(self):
        self.function_name = None
        self.records = []

    def visit_FunctionDef(self, node):
        previous = self.function_name
        self.function_name = node.name
        self.generic_visit(node)
        self.function_name = previous

    visit_AsyncFunctionDef = visit_FunctionDef

    def visit_Call(self, node):
        function_name = getattr(node.func, "id", None)
        if function_name == "agent_model_settings" and len(node.args) >= 2:
            role_node = node.args[1]
            role = role_node.value if isinstance(role_node, ast.Constant) else None
            values = {"temperature": None, "max_tokens": None}
            for keyword in node.keywords:
                if keyword.arg in values and isinstance(keyword.value, ast.Constant):
                    values[keyword.arg] = keyword.value.value
            self.records.append(
                {
                    "builder": self.function_name,
                    "role": role,
                    "temperature": values["temperature"],
                    "max_tokens": values["max_tokens"],
                    "source_line": node.lineno,
                }
            )
        self.generic_visit(node)


def extract_agent_parameters():
    path = PROJECT_DIR / "src" / "table2text" / "agents.py"
    visitor = AgentParameterVisitor()
    visitor.visit(ast.parse(path.read_text(encoding="utf-8")))
    return visitor.records


current_implementation_sha256, current_file_manifest = implementation_fingerprint()
variant_sha256 = sha256_file(VARIANT_PATH)
prepared_examples_sha256 = sha256_file(PATHS["prepared_examples"])

if not FREEZE_MANIFEST_PATH.exists():
    snapshot_file_root = SNAPSHOT_DIR / "files"
    for item, source in zip(current_file_manifest, implementation_files()):
        destination = snapshot_file_root / item["path"]
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

    freeze_manifest = {
        "experiment_id": EXPERIMENT_ID,
        "frozen_at": utc_now(),
        "freeze_basis": "exact_source_tree_snapshot",
        "git_commit": git_output("rev-parse", "HEAD"),
        "git_branch": git_output("branch", "--show-current"),
        "git_status_porcelain": (git_output("status", "--porcelain") or "").splitlines(),
        "git_worktree_clean": not bool(git_output("status", "--porcelain")),
        "implementation_sha256": current_implementation_sha256,
        "implementation_files": current_file_manifest,
        "variant_path": relative_path(VARIANT_PATH),
        "variant_sha256": variant_sha256,
        "prepared_examples_path": relative_path(PATHS["prepared_examples"]),
        "prepared_examples_sha256": prepared_examples_sha256,
        "models": {role.removesuffix("_model"): MODEL for role in MODEL_FIELDS},
        "generation_seed": GENERATION_SEED,
        "resolved_settings": frozen_overrides,
        "agent_model_parameters": extract_agent_parameters(),
        "provider_seed_forwarded_to_deepseek": False,
        "deepseek_base_url": os.getenv("DEEPSEEK_BASE_URL", "provider default"),
        "runtime": {
            "python": sys.version,
            "platform": platform.platform(),
            "table2text_package_source": str(PROJECT_DIR / "src" / "table2text"),
            "pydantic": package_version("pydantic"),
            "pydantic_ai": package_version("pydantic-ai"),
            "openai": package_version("openai"),
            "pandas": package_version("pandas"),
        },
        "secrets_recorded": False,
    }
    write_json_atomic(FREEZE_MANIFEST_PATH, freeze_manifest)
    log(f"Frozen implementation snapshot: {current_implementation_sha256}")
else:
    freeze_manifest = read_json(FREEZE_MANIFEST_PATH)
    log(f"Reusing frozen implementation: {freeze_manifest['implementation_sha256']}")


def assert_frozen_state():
    current_sha256, _ = implementation_fingerprint()
    errors = []
    if current_sha256 != freeze_manifest["implementation_sha256"]:
        errors.append(
            "implementation fingerprint changed after protected selection"
        )
    if sha256_file(VARIANT_PATH) != freeze_manifest["variant_sha256"]:
        errors.append("protected variant configuration changed")
    if sha256_file(PATHS["prepared_examples"]) != freeze_manifest["prepared_examples_sha256"]:
        errors.append("prepared benchmark file changed")
    if errors:
        raise RuntimeError("Protected-run freeze violation: " + "; ".join(errors))
    return current_sha256


assert_frozen_state()

print("Freeze basis:", freeze_manifest["freeze_basis"])
print("Git commit:", freeze_manifest["git_commit"])
print("Git worktree clean:", freeze_manifest["git_worktree_clean"])
print("Implementation SHA-256:", freeze_manifest["implementation_sha256"])
print("Variant SHA-256:", freeze_manifest["variant_sha256"])
print("Models:", sorted(set(freeze_manifest["models"].values())))
display(pd.DataFrame(freeze_manifest["agent_model_parameters"]))




[01:28:22] Created frozen variant: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_full_system/config/protected_full_system_flash.json
[01:28:22] Frozen implementation snapshot: f525ca0c932819f533eab9229752a055a8e9c1acef8dea420bfa7e1e8b6b91fb
Freeze basis: exact_source_tree_snapshot
Git commit: 4a333a76702e6c056d8364dc212ce360d3fb1b92
Git worktree clean: False
Implementation SHA-256: f525ca0c932819f533eab9229752a055a8e9c1acef8dea420bfa7e1e8b6b91fb
Variant SHA-256: 1fe2242bf777f26ada83114301c7c1fc11b6c25f613fa9d42e0b9484e4555d85
Models: ['deepseek:deepseek-v4-flash']


,builder,role,temperature,max_tokens,source_line
0,build_data_understanding_agent,data_understanding,0.00,7000,286
1,build_orchestrator_agent,orchestrator,0.10,8000,488
2,build_evidence_agent,evidence,0.00,9000,883
3,build_verifier_agent,verifier,0.00,8000,1002
4,build_insight_synthesis_agent,evidence,0.00,8000,1939
5,build_insight_verifier_agent,verifier,0.00,8000,2145
6,build_writer_agent,writer,0.15,11000,2968
7,build_auditor_agent,auditor,0.10,12000,3522


## 2. Freeze the unseen 25-example selection

Selection uses only dataset ID and example ID. Every generation record found
elsewhere under `evaluation/**/generations/**` is excluded. Sources and
references do not participate in selection. The five datasets are then
interleaved, so a partial run remains balanced across task families.

The operational examples replace the true references with a fixed sentinel
before any generator call. True references are recovered only in the final,
explicitly opt-in post-generation metric phase.



In [4]:
TARGET_DATASET_SET = set(DATASETS)
HELD_OUT_REFERENCE_SENTINEL = "<HELD_OUT_REFERENCE_NOT_AVAILABLE_DURING_GENERATION>"
REFERENCE_LIKE_METADATA_KEYS = {
    "answer",
    "answers",
    "description",
    "descriptions",
    "gold",
    "news_article",
    "ref",
    "reference",
    "references",
    "summary",
    "summaries",
    "target",
    "targets",
    "text",
}


def historical_generation_inventory():
    identities = defaultdict(set)
    files = []
    evaluation_root = PROJECT_DIR / "evaluation"
    for path in sorted(evaluation_root.rglob("*.jsonl")):
        if path.is_relative_to(ARTIFACT_DIR):
            continue
        if "generations" not in path.parts:
            continue

        matched = 0
        for item in read_jsonl_objects(path):
            if "generated_text" not in item or "variant_id" not in item:
                continue
            dataset_id = str(item.get("dataset_id", ""))
            example_id = item.get("example_id")
            if dataset_id in TARGET_DATASET_SET and example_id is not None:
                identities[dataset_id].add(str(example_id))
                matched += 1
        if matched:
            files.append(
                {
                    "path": relative_path(path),
                    "sha256": sha256_file(path),
                    "matching_generation_rows": matched,
                }
            )
    return identities, files


all_examples = read_examples(PATHS["prepared_examples"])
example_lookup = {
    (str(example.dataset_id), str(example.example_id)): example
    for example in all_examples
}

if not SELECTION_MANIFEST_PATH.exists():
    assert_frozen_state()
    historical_ids, historical_files = historical_generation_inventory()
    selected_by_dataset = {}

    for dataset_id in DATASETS:
        candidates = sorted(
            [
                example
                for example in all_examples
                if example.dataset_id == dataset_id
                and str(example.example_id) not in historical_ids[dataset_id]
            ],
            key=lambda item: str(item.example_id),
        )
        if len(candidates) < EXAMPLES_PER_DATASET:
            raise RuntimeError(
                f"{dataset_id} has only {len(candidates)} previously unused "
                f"prepared examples; {EXAMPLES_PER_DATASET} are required."
            )
        generator = random.Random(f"{SELECTION_SEED}:{dataset_id}")
        selected_by_dataset[dataset_id] = sorted(
            generator.sample(candidates, EXAMPLES_PER_DATASET),
            key=lambda item: str(item.example_id),
        )

    # Round-robin order: one example from each dataset per block.
    selected_examples = [
        selected_by_dataset[dataset_id][position]
        for position in range(EXAMPLES_PER_DATASET)
        for dataset_id in DATASETS
    ]
    selection_timestamp = utc_now()

    exclusion_manifest = {
        "created_at": selection_timestamp,
        "selection_rule": (
            "Exclude every prior GenerationRecord found outside this protected "
            "experiment; sample by identity only."
        ),
        "historical_generation_files": historical_files,
        "excluded_example_ids": {
            dataset_id: sorted(historical_ids[dataset_id])
            for dataset_id in DATASETS
        },
    }
    write_json_atomic(EXCLUSION_MANIFEST_PATH, exclusion_manifest)

    selection_manifest = {
        "experiment_id": EXPERIMENT_ID,
        "selected_at": selection_timestamp,
        "protected": True,
        "selection_seed": SELECTION_SEED,
        "selection_algorithm": (
            "Dataset-stratified random sample after historical-generation "
            "exclusion; source and reference fields were not used."
        ),
        "examples_per_dataset": EXAMPLES_PER_DATASET,
        "dataset_order": DATASETS,
        "run_order": [
            {
                "position": index,
                "dataset_id": example.dataset_id,
                "example_id": str(example.example_id),
                "source_sha256": example.source_sha256,
                "reference_sha256": example.reference_sha256,
                "task_family": example.task_family.value,
                "output_mode": example.output_mode.value,
                "language": example.language,
            }
            for index, example in enumerate(selected_examples, start=1)
        ],
        "historical_overlap_count_at_selection": 0,
        "researcher_declared_no_prior_manual_inspection": (
            RESEARCHER_CONFIRMS_NO_PRIOR_MANUAL_INSPECTION
        ),
        "freeze_manifest_sha256": sha256_file(FREEZE_MANIFEST_PATH),
    }
    write_json_atomic(SELECTION_MANIFEST_PATH, selection_manifest)
    log(f"Protected selection frozen at {selection_timestamp}")
else:
    selection_manifest = read_json(SELECTION_MANIFEST_PATH)
    selected_examples = []
    for item in selection_manifest["run_order"]:
        key = (item["dataset_id"], str(item["example_id"]))
        if key not in example_lookup:
            raise RuntimeError(f"Selected example disappeared from prepared data: {key}")
        example = example_lookup[key]
        if example.source_sha256 != item["source_sha256"]:
            raise RuntimeError(f"Source hash changed for selected example: {key}")
        if example.reference_sha256 != item["reference_sha256"]:
            raise RuntimeError(f"Reference hash changed for selected example: {key}")
        selected_examples.append(example)
    log(f"Reusing protected selection from {selection_manifest['selected_at']}")

expected_total = len(DATASETS) * EXAMPLES_PER_DATASET
assert len(selected_examples) == expected_total
assert Counter(example.dataset_id for example in selected_examples) == Counter(
    {dataset_id: EXAMPLES_PER_DATASET for dataset_id in DATASETS}
)


def sanitize_operational_metadata(metadata):
    return {
        key: value
        for key, value in metadata.items()
        if key.casefold() not in REFERENCE_LIKE_METADATA_KEYS
        and "reference" not in key.casefold()
        and not key.casefold().startswith("target")
    }


operational_examples = [
    example.model_copy(
        update={
            "references": [HELD_OUT_REFERENCE_SENTINEL],
            "metadata": sanitize_operational_metadata(example.metadata),
        }
    )
    for example in selected_examples
]
write_jsonl_atomic(OPERATIONAL_EXAMPLES_PATH, operational_examples)


def audit_materialized_model_input(example):
    path = materialise_input(example, MODEL_INPUT_AUDIT_DIR)
    payload = json.loads(path.read_text(encoding="utf-8"))
    serialized = json.dumps(payload, ensure_ascii=False)
    forbidden_top_level = {
        "references",
        "reference_sha256",
        "target",
        "targets",
        "summary",
        "summaries",
    }
    present = sorted(forbidden_top_level & set(payload)) if isinstance(payload, dict) else []
    if present:
        raise RuntimeError(
            f"Reference-like top-level fields reached model input for "
            f"{example.dataset_id}/{example.example_id}: {present}"
        )
    if HELD_OUT_REFERENCE_SENTINEL in serialized:
        raise RuntimeError(
            f"Held-out reference sentinel reached model input for "
            f"{example.dataset_id}/{example.example_id}."
        )
    return {
        "dataset_id": example.dataset_id,
        "example_id": str(example.example_id),
        "materialized_input_path": relative_path(path),
        "materialized_input_sha256": sha256_file(path),
        "reference_fields_present": present,
        "reference_sentinel_present": False,
    }


model_input_audits = [
    audit_materialized_model_input(example) for example in operational_examples
]
write_json_atomic(
    MODEL_INPUT_AUDIT_DIR / "model_input_isolation_manifest.json",
    {
        "created_at": utc_now(),
        "reference_isolation": "pass",
        "records": model_input_audits,
    },
)

selected_table = pd.DataFrame(selection_manifest["run_order"])
display(
    selected_table[
        [
            "position",
            "dataset_id",
            "example_id",
            "task_family",
            "output_mode",
            "language",
        ]
    ]
)
print("Selected examples:", len(selected_examples))
print("Historical overlap:", selection_manifest["historical_overlap_count_at_selection"])
print("Reference isolation: PASS (sentinel absent from all materialized model inputs)")




[01:28:22] Protected selection frozen at 2026-08-21T00:28:22.944030+00:00


,position,dataset_id,example_id,task_family,output_mode,language
0,1,e2e_nlg,e2e_nlg-test-1330,attribute_verbalisation,short_text,en
1,2,web_nlg,web_nlg_en-test-1209,triple_verbalisation,short_text,en
2,3,dart,dart-test-1791,triple_verbalisation,short_text,en
3,4,totto,totto-validation-1828,highlighted_table_description,one_sentence,en
4,5,sportsett_basketball,5130,event_report,multi_paragraph_report,en
5,6,e2e_nlg,e2e_nlg-test-209,attribute_verbalisation,short_text,en
6,7,web_nlg,web_nlg_en-test-1330,triple_verbalisation,short_text,en
7,8,dart,dart-test-1805,triple_verbalisation,short_text,en
8,9,totto,totto-validation-4467,highlighted_table_description,one_sentence,en
9,10,sportsett_basketball,5372,event_report,multi_paragraph_report,en


Selected examples: 25
Historical overlap: 0
Reference isolation: PASS (sentinel absent from all materialized model inputs)


## 3. Initialize the experiment-level manifest

This manifest is updated after every case. It records the frozen code and
config, protected selection, model roles, task contracts, reference boundary,
completion counts, and whether any implementation drift was detected.



In [5]:
def collect_final_records():
    records = []
    for item in selection_manifest["run_order"]:
        slug = f"{safe_name(item['dataset_id'])}__{safe_name(item['example_id'])}"
        path = SHARD_DIR / f"{slug}.jsonl"
        if not path.exists():
            continue
        rows = read_generations(path)
        if rows:
            records.append(rows[-1])
    write_jsonl_atomic(SEALED_GENERATIONS_PATH, records)
    return records


def update_batch_manifest(status=None):
    records = collect_final_records()
    successful = [record for record in records if not record.error and record.generated_text]
    failed = [record for record in records if record.error or not record.generated_text]
    inferred_status = (
        "complete"
        if len(successful) == expected_total and not failed
        else "running"
        if records
        else "selected"
    )
    prior = read_json(BATCH_MANIFEST_PATH, {})
    payload = {
        **prior,
        "experiment": "Protected Holdout Validation",
        "experiment_id": EXPERIMENT_ID,
        "selection_date": selection_manifest["selected_at"],
        "last_updated": utc_now(),
        "status": status or inferred_status,
        "protected_unseen": True,
        "system_frozen_before_selection": True,
        "freeze_basis": freeze_manifest["freeze_basis"],
        "git_commit": freeze_manifest["git_commit"],
        "git_worktree_clean_at_freeze": freeze_manifest["git_worktree_clean"],
        "implementation_sha256": freeze_manifest["implementation_sha256"],
        "configuration_path": relative_path(VARIANT_PATH),
        "configuration_sha256": freeze_manifest["variant_sha256"],
        "selection_manifest_sha256": sha256_file(SELECTION_MANIFEST_PATH),
        "model_input_isolation_manifest_sha256": sha256_file(
            MODEL_INPUT_AUDIT_DIR / "model_input_isolation_manifest.json"
        ),
        "model": "DeepSeek V4 Flash",
        "models_by_role": freeze_manifest["models"],
        "model_parameters": freeze_manifest["agent_model_parameters"],
        "generation_seed": GENERATION_SEED,
        "provider_seed_forwarded": False,
        "examples": expected_total,
        "datasets": {
            dataset_id: EXAMPLES_PER_DATASET for dataset_id in DATASETS
        },
        "selected_examples": selection_manifest["run_order"],
        "previous_manual_inspection_of_selected_examples": (
            "none_declared"
            if RESEARCHER_CONFIRMS_NO_PRIOR_MANUAL_INSPECTION
            else "not_confirmed"
        ),
        "historical_generation_overlap_count": (
            selection_manifest["historical_overlap_count_at_selection"]
        ),
        "development_changes_after_holdout_selection": (
            "none_verified_by_fingerprint"
            if RESEARCHER_CONFIRMS_NO_DEVELOPMENT_CHANGES_AFTER_SELECTION
            else "not_confirmed"
        ),
        "implementation_drift_detected": False,
        "human_references_available_to_generator": False,
        "reference_isolation_method": (
            "True references replaced by a sentinel in BenchmarkExample before "
            "generation; materialized model inputs contain neither references nor "
            "the sentinel."
        ),
        "released_generations_per_example": 1,
        "generation_configuration": relative_path(VARIANT_PATH),
        "completion": {
            "successful": len(successful),
            "failed": len(failed),
            "not_started": expected_total - len(records),
        },
        "sealed_generations_path": relative_path(SEALED_GENERATIONS_PATH),
        "summary_path": relative_path(SUMMARY_JSONL_PATH),
        "progress_log_path": relative_path(PROGRESS_LOG_PATH),
        "true_references_unsealed_at": prior.get("true_references_unsealed_at"),
    }
    if payload["status"] == "complete" and "generation_completed_at" not in payload:
        payload["generation_completed_at"] = utc_now()
    if records and "generation_started_at" not in payload:
        payload["generation_started_at"] = utc_now()
    write_json_atomic(BATCH_MANIFEST_PATH, payload)
    return payload


batch_manifest = update_batch_manifest()
display(pd.DataFrame([batch_manifest["completion"]]))




,successful,failed,not_started
0,0,0,25


## 4. Artifact and token extraction

These functions summarize the workflow artifacts themselves. They do not
infer missing information. Provider-reported token usage is parsed from the
persisted trace, and monetary cost remains null unless directly returned.



In [6]:
USAGE_FIELD_PATTERN = {
    "input_tokens": re.compile(r"\binput_tokens=(\d+)"),
    "cache_read_tokens": re.compile(r"\bcache_read_tokens=(\d+)"),
    "output_tokens": re.compile(r"\boutput_tokens=(\d+)"),
    "requests": re.compile(r"\brequests=(\d+)"),
}


def parse_usage_string(value):
    if not isinstance(value, str):
        return None
    result = {}
    for name, pattern in USAGE_FIELD_PATTERN.items():
        match = pattern.search(value)
        result[name] = int(match.group(1)) if match else 0
    if not any(result.values()):
        return None
    result["total_tokens"] = result["input_tokens"] + result["output_tokens"]
    return result


def locate_pipeline_result(record):
    if record.pipeline_result_path:
        path = Path(record.pipeline_result_path)
        if path.exists():
            return path
    if record.run_id:
        matches = list(RUN_ROOT.rglob(f"{record.run_id}/pipeline_result.json"))
        if len(matches) == 1:
            return matches[0]
    return None


def stage_usage_rows(record, run_directory):
    trace_path = run_directory / "trace.jsonl"
    rows = []
    for index, event in enumerate(read_jsonl_objects(trace_path), start=1):
        usage = parse_usage_string(event.get("details", {}).get("usage"))
        if usage is None:
            continue
        rows.append(
            {
                "dataset_id": record.dataset_id,
                "example_id": str(record.example_id),
                "run_id": record.run_id,
                "trace_event": index,
                "timestamp": event.get("timestamp"),
                "stage": event.get("stage"),
                "status": event.get("status"),
                **usage,
            }
        )
    return rows


def run_artifact_index(record, run_directory):
    files = []
    for path in sorted(run_directory.rglob("*")):
        if path.is_file():
            files.append(
                {
                    "path": relative_path(path),
                    "sha256": sha256_file(path),
                    "bytes": path.stat().st_size,
                }
            )
    return {
        "dataset_id": record.dataset_id,
        "example_id": str(record.example_id),
        "run_id": record.run_id,
        "run_directory": relative_path(run_directory),
        "artifact_count": len(files),
        "artifacts": files,
        "index_sha256": canonical_sha256(files),
    }


def trace_summary(run_directory):
    events = read_jsonl_objects(run_directory / "trace.jsonl")
    fallbacks = [event for event in events if event.get("status") == "fallback"]
    retries = [event for event in events if ".retry." in str(event.get("stage", ""))]
    non_success = [
        event
        for event in events
        if event.get("status") in {"fallback", "error", "failed"}
    ]
    return events, fallbacks, retries, non_success


def extract_summary(record):
    pipeline_path = locate_pipeline_result(record)
    if pipeline_path is None:
        return {
            "dataset_id": record.dataset_id,
            "example_id": str(record.example_id),
            "run_id": record.run_id,
            "execution_outcome": "failed",
            "error": record.error or "pipeline_result.json not found",
            "exact_final_output": record.generated_text,
        }, [], [], None

    result = read_json(pipeline_path)
    run_directory = pipeline_path.parent
    run_manifest = read_json(run_directory / "00_manifest.json", {})
    events, fallbacks, retries, non_success = trace_summary(run_directory)
    usage_rows = stage_usage_rows(record, run_directory)
    artifact_index = run_artifact_index(record, run_directory)

    fact_candidates = nested_list(result, "fact_candidates", "candidates")
    writer_ready_facts = nested_list(result, "fact_ledger", "writer_ready_facts")
    rejected_facts = nested_list(result, "fact_ledger", "rejected_facts")
    evidence_items = nested_list(result, "evidence_ledger", "items")
    verified_insights = nested_list(result, "insight_ledger", "verified_insights")
    rejected_insights = nested_list(result, "insight_ledger", "rejected_insights")
    unverified_insights = nested_list(result, "insight_ledger", "unverified_insights")
    hypothesis_insights = nested_list(result, "insight_ledger", "hypothesis_only_insights")
    insight_candidate_payload = read_json(run_directory / "07_insight_candidates.json", {})
    insight_candidates = (
        insight_candidate_payload.get("candidates", [])
        if isinstance(insight_candidate_payload, dict)
        else []
    )

    final_output = result.get("final_writer_output", {})
    raw_output = result.get("raw_writer_output", {})
    final_audit = result.get("final_audit", {})
    support = final_output.get("sentence_support", []) or []
    repair_rounds = result.get("repair_rounds_used", 0) or 0
    final_writer_mode = final_output.get("writer_mode") or record.writer_mode
    raw_writer_mode = raw_output.get("writer_mode")
    used_deterministic_fallback = "deterministic" in str(raw_writer_mode).casefold()

    if repair_rounds > 0 or final_audit.get("applied_patches"):
        final_generation_path = "auditor_repaired"
    elif used_deterministic_fallback:
        final_generation_path = "deterministic_fallback"
    else:
        final_generation_path = "normal_llm_writer"

    start_time = run_manifest.get("created_at")
    end_time = events[-1].get("timestamp") if events else None
    total_input_tokens = sum(row["input_tokens"] for row in usage_rows)
    total_output_tokens = sum(row["output_tokens"] for row in usage_rows)
    total_cache_tokens = sum(row["cache_read_tokens"] for row in usage_rows)
    total_requests = sum(row["requests"] for row in usage_rows)

    final_report_path = run_directory / "final_report.md"
    exact_released_report = (
        final_report_path.read_text(encoding="utf-8")
        if final_report_path.exists()
        else record.generated_text
    )
    actual_input_path = None
    input_paths = run_manifest.get("input_paths", [])
    if input_paths:
        candidate = Path(input_paths[0])
        if candidate.exists():
            actual_input_path = candidate
    expected_input_audit = next(
        (
            item
            for item in model_input_audits
            if item["dataset_id"] == record.dataset_id
            and item["example_id"] == str(record.example_id)
        ),
        None,
    )
    actual_input_sha256 = (
        sha256_file(actual_input_path) if actual_input_path is not None else None
    )
    expected_input_sha256 = (
        expected_input_audit["materialized_input_sha256"]
        if expected_input_audit is not None
        else None
    )

    summary = {
        "dataset_id": record.dataset_id,
        "example_id": str(record.example_id),
        "protected_unseen": True,
        "selection_timestamp": selection_manifest["selected_at"],
        "run_id": record.run_id,
        "generation_id": record.generation_id,
        "execution_outcome": "success" if not record.error else "failed",
        "error": record.error,
        "start_time": start_time,
        "end_time": end_time,
        "elapsed_seconds": record.elapsed_seconds,
        "task_family": record.task_family.value,
        "request": record.request,
        "output_mode": record.output_mode.value,
        "language": record.language,
        "task_contract": run_manifest.get("task_contract"),
        "report_genre": run_manifest.get("report_genre"),
        "communication_task": run_manifest.get("communication_task"),
        "focus_scope": run_manifest.get("focus_scope"),
        "models": run_manifest.get("models"),
        "generation_seed": record.seed,
        "configuration_sha256": freeze_manifest["variant_sha256"],
        "implementation_sha256": freeze_manifest["implementation_sha256"],
        "reference_available_to_generator": False,
        "source_sha256": next(
            item["source_sha256"]
            for item in selection_manifest["run_order"]
            if item["dataset_id"] == record.dataset_id
            and item["example_id"] == str(record.example_id)
        ),
        "reference_sha256": next(
            item["reference_sha256"]
            for item in selection_manifest["run_order"]
            if item["dataset_id"] == record.dataset_id
            and item["example_id"] == str(record.example_id)
        ),
        "input_paths": input_paths,
        "materialized_model_input_sha256": actual_input_sha256,
        "model_input_matches_reference_isolation_audit": (
            actual_input_sha256 is not None
            and actual_input_sha256 == expected_input_sha256
        ),
        "pipeline_result_path": relative_path(pipeline_path),
        "run_directory": relative_path(run_directory),
        "final_report_path": relative_path(final_report_path),
        "initial_writer_mode": raw_writer_mode,
        "final_writer_mode": final_writer_mode,
        "used_deterministic_fallback": used_deterministic_fallback,
        "final_generation_path": final_generation_path,
        "repair_rounds_used": repair_rounds,
        "audit_decision": final_audit.get("decision"),
        "release_status": result.get("release_status") or record.release_status,
        "approved_for_release": result.get("approved_for_release"),
        "primary_evaluation_eligible": result.get("primary_evaluation_eligible"),
        "native_support_rate": final_audit.get("support_rate"),
        "factual_sentence_count": final_audit.get("factual_sentence_count"),
        "supported_sentence_count": final_audit.get("supported_sentence_count"),
        "unsupported_factual_sentence_count": (
            (final_audit.get("factual_sentence_count") or 0)
            - (final_audit.get("supported_sentence_count") or 0)
        ),
        "sentence_support_mapping_count": len(support),
        "output_word_count": markdown_word_count(record.generated_text),
        "output_sentence_count": prose_sentence_count(record.generated_text),
        "evidence_item_count": len(evidence_items),
        "fact_candidate_count": len(fact_candidates),
        "verified_fact_count": len(writer_ready_facts),
        "rejected_fact_count": len(rejected_facts),
        "insight_candidate_count": len(insight_candidates),
        "verified_insight_count": len(verified_insights),
        "rejected_insight_count": len(rejected_insights),
        "unverified_insight_count": len(unverified_insights),
        "hypothesis_only_insight_count": len(hypothesis_insights),
        "top_level_attempt_count": sum(
            1
            for item in read_jsonl_objects(ATTEMPT_LOG_PATH)
            if item.get("event") == "started"
            and item.get("dataset_id") == record.dataset_id
            and item.get("example_id") == str(record.example_id)
        ),
        "trace_retry_event_count": len(retries),
        "fallback_event_count": len(fallbacks),
        "non_success_trace_event_count": len(non_success),
        "fallbacks": fallbacks,
        "trace_retries": retries,
        "provider_reported_input_tokens": total_input_tokens,
        "provider_reported_output_tokens": total_output_tokens,
        "provider_reported_total_tokens": total_input_tokens + total_output_tokens,
        "provider_reported_cache_read_tokens": total_cache_tokens,
        "provider_reported_requests": total_requests,
        "monetary_cost": record.estimated_cost_gbp,
        "monetary_cost_source": (
            "provider/evaluation record"
            if record.estimated_cost_gbp is not None
            else "not directly available"
        ),
        "artifact_count": artifact_index["artifact_count"],
        "artifact_index_sha256": artifact_index["index_sha256"],
        "exact_final_output": record.generated_text,
        "exact_released_report": exact_released_report,
    }

    support_rows = [
        {
            "dataset_id": record.dataset_id,
            "example_id": str(record.example_id),
            "run_id": record.run_id,
            **mapping,
        }
        for mapping in support
    ]
    return summary, usage_rows, support_rows, artifact_index


def csv_safe_summary(summary):
    excluded = {
        "task_contract",
        "models",
        "input_paths",
        "fallbacks",
        "trace_retries",
        "exact_final_output",
        "exact_released_report",
    }
    return {key: value for key, value in summary.items() if key not in excluded}


def escape_markdown(value):
    if value is None:
        return ""
    return str(value).replace("|", "\\|").replace("\n", " ")


def rebuild_aggregate_artifacts():
    records = collect_final_records()
    summaries = []
    usage_rows = []
    support_rows = []
    artifact_indexes = []
    for record in records:
        summary, usage, support, artifact_index = extract_summary(record)
        summaries.append(summary)
        usage_rows.extend(usage)
        support_rows.extend(support)
        if artifact_index is not None:
            artifact_indexes.append(artifact_index)

    write_jsonl_atomic(SUMMARY_JSONL_PATH, summaries)
    write_jsonl_atomic(SUPPORT_MAP_JSONL_PATH, support_rows)
    write_jsonl_atomic(ARTIFACT_INDEX_PATH, artifact_indexes)
    pd.DataFrame([csv_safe_summary(item) for item in summaries]).to_csv(
        SUMMARY_CSV_PATH,
        index=False,
    )
    pd.DataFrame(usage_rows).to_csv(STAGE_USAGE_CSV_PATH, index=False)

    manifest = update_batch_manifest()
    lines = [
        "# Protected Holdout Execution Report",
        "",
        f"- Experiment: `{EXPERIMENT_ID}`",
        f"- Selection timestamp: `{selection_manifest['selected_at']}`",
        f"- Frozen implementation: `{freeze_manifest['implementation_sha256']}`",
        f"- Git commit: `{freeze_manifest['git_commit']}`",
        f"- Configuration: `{relative_path(VARIANT_PATH)}`",
        f"- Configuration SHA-256: `{freeze_manifest['variant_sha256']}`",
        f"- Model for all six roles: `{MODEL}`",
        f"- Status: `{manifest['status']}`",
        f"- Completion: `{manifest['completion']}`",
        "- References available during generation: `No`",
        "",
        "## Run Summary",
        "",
        "| Dataset | Example | Outcome | Path | Release | Support | Words | Tokens | Seconds |",
        "|---|---|---|---|---|---:|---:|---:|---:|",
    ]
    for item in summaries:
        lines.append(
            "| "
            + " | ".join(
                [
                    escape_markdown(item.get("dataset_id")),
                    escape_markdown(item.get("example_id")),
                    escape_markdown(item.get("execution_outcome")),
                    escape_markdown(item.get("final_generation_path")),
                    escape_markdown(item.get("release_status")),
                    escape_markdown(item.get("native_support_rate")),
                    escape_markdown(item.get("output_word_count")),
                    escape_markdown(item.get("provider_reported_total_tokens")),
                    escape_markdown(
                        round(item.get("elapsed_seconds") or 0.0, 1)
                    ),
                ]
            )
            + " |"
        )

    if usage_rows:
        usage_frame = pd.DataFrame(usage_rows)
        stage_totals = (
            usage_frame.groupby("stage", as_index=False)[
                ["input_tokens", "output_tokens", "total_tokens", "requests"]
            ]
            .sum()
            .sort_values("total_tokens", ascending=False)
        )
        lines.extend(
            [
                "",
                "## Provider-Reported Usage by Stage",
                "",
                "| Stage | Input tokens | Output tokens | Total tokens | Requests |",
                "|---|---:|---:|---:|---:|",
            ]
        )
        for row in stage_totals.to_dict(orient="records"):
            lines.append(
                f"| {escape_markdown(row['stage'])} | {row['input_tokens']} | "
                f"{row['output_tokens']} | {row['total_tokens']} | {row['requests']} |"
            )

    lines.extend(
        [
            "",
            "## Artifact Locations",
            "",
            f"- Batch manifest: `{relative_path(BATCH_MANIFEST_PATH)}`",
            f"- Exact outputs and run summaries: `{relative_path(SUMMARY_JSONL_PATH)}`",
            f"- Sentence support: `{relative_path(SUPPORT_MAP_JSONL_PATH)}`",
            f"- Stage usage: `{relative_path(STAGE_USAGE_CSV_PATH)}`",
            f"- Run checksums: `{relative_path(ARTIFACT_INDEX_PATH)}`",
            f"- Progress log: `{relative_path(PROGRESS_LOG_PATH)}`",
            "",
        ]
    )
    EXECUTION_REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
    return pd.DataFrame([csv_safe_summary(item) for item in summaries])


summary_frame = rebuild_aggregate_artifacts()
if not summary_frame.empty:
    display(summary_frame)




## 5. Run or resume the protected generation batch

Each example has an independent generation shard. Failed attempts are copied
to `attempt_records/` before retry; successful outputs are never regenerated.
The exact implementation/config/input fingerprints are checked immediately
before every call.



In [7]:
def operational_example_lookup():
    return {
        (str(example.dataset_id), str(example.example_id)): example
        for example in operational_examples
    }


def attempt_count(dataset_id, example_id):
    return sum(
        1
        for item in read_jsonl_objects(ATTEMPT_LOG_PATH)
        if item.get("event") == "started"
        and item.get("dataset_id") == dataset_id
        and item.get("example_id") == str(example_id)
    )


def existing_success(shard_path):
    if not shard_path.exists():
        return None
    rows = read_generations(shard_path)
    if not rows:
        return None
    row = rows[-1]
    return row if not row.error and bool(row.generated_text.strip()) else None


async def run_protected_batch():
    operational_lookup = operational_example_lookup()
    completed_before = {
        (record.dataset_id, str(record.example_id))
        for record in collect_final_records()
        if not record.error and record.generated_text.strip()
    }
    pending = [
        item
        for item in selection_manifest["run_order"]
        if (item["dataset_id"], str(item["example_id"])) not in completed_before
    ]
    if MAX_CASES_THIS_SESSION is not None:
        pending = pending[:MAX_CASES_THIS_SESSION]

    log("=" * 80)
    log(
        f"Protected batch: {len(completed_before)}/{expected_total} already complete; "
        f"{len(pending)} scheduled this session."
    )
    if pending:
        manifest = read_json(BATCH_MANIFEST_PATH, {})
        if not manifest.get("generation_started_at"):
            manifest["generation_started_at"] = utc_now()
            manifest["status"] = "running"
            manifest["last_updated"] = utc_now()
            write_json_atomic(BATCH_MANIFEST_PATH, manifest)

    for session_index, item in enumerate(pending, start=1):
        dataset_id = item["dataset_id"]
        example_id = str(item["example_id"])
        key = (dataset_id, example_id)
        example = operational_lookup[key]
        slug = f"{safe_name(dataset_id)}__{safe_name(example_id)}"
        example_path = PREPARED_DIR / "shards" / f"{slug}.jsonl"
        shard_path = SHARD_DIR / f"{slug}.jsonl"
        run_root = RUN_ROOT / slug
        example_path.parent.mkdir(parents=True, exist_ok=True)
        write_jsonl_atomic(example_path, [example])

        success = existing_success(shard_path)
        if success is not None:
            log(f"SKIP {dataset_id}/{example_id}: successful shard already exists")
            continue

        prior_attempts = attempt_count(dataset_id, example_id)
        remaining_attempts = max(
            0,
            MAX_TOP_LEVEL_ATTEMPTS_PER_EXAMPLE - prior_attempts,
        )
        if remaining_attempts == 0:
            log(f"EXHAUSTED {dataset_id}/{example_id}: no attempts remain")
            continue

        log("-" * 80)
        log(
            f"CASE {session_index}/{len(pending)} this session; "
            f"protected position {item['position']}/{expected_total}: "
            f"{dataset_id}/{example_id}"
        )

        case_succeeded = False
        for _ in range(remaining_attempts):
            assert_frozen_state()
            attempt_number = attempt_count(dataset_id, example_id) + 1
            attempt_started = utc_now()
            append_jsonl(
                ATTEMPT_LOG_PATH,
                {
                    "event": "started",
                    "timestamp": attempt_started,
                    "dataset_id": dataset_id,
                    "example_id": example_id,
                    "attempt": attempt_number,
                    "implementation_sha256": freeze_manifest["implementation_sha256"],
                    "configuration_sha256": freeze_manifest["variant_sha256"],
                },
            )

            if shard_path.exists():
                archived = ATTEMPT_RECORD_DIR / f"{slug}__before_attempt_{attempt_number}.jsonl"
                shutil.copy2(shard_path, archived)
                shard_path.unlink()

            try:
                frame = await with_heartbeat(
                    generate_reports_for_notebook(
                        PROJECT_DIR,
                        examples_path=example_path,
                        variants_path=VARIANT_PATH,
                        output_path=shard_path,
                        run_root=run_root,
                        resume=False,
                    ),
                    label=(
                        f"{dataset_id}/{example_id} attempt "
                        f"{attempt_number}/{MAX_TOP_LEVEL_ATTEMPTS_PER_EXAMPLE}"
                    ),
                )
                row = frame.iloc[-1].to_dict() if not frame.empty else {}
                error = row.get("error") or None
                generated_text = str(row.get("generated_text") or "")
                success = not error and bool(generated_text.strip())
                append_jsonl(
                    ATTEMPT_LOG_PATH,
                    {
                        "event": "finished",
                        "timestamp": utc_now(),
                        "dataset_id": dataset_id,
                        "example_id": example_id,
                        "attempt": attempt_number,
                        "success": success,
                        "error": error,
                        "run_id": row.get("run_id"),
                        "pipeline_result_path": row.get("pipeline_result_path"),
                        "elapsed_seconds": row.get("elapsed_seconds"),
                    },
                )
                if shard_path.exists():
                    shutil.copy2(
                        shard_path,
                        ATTEMPT_RECORD_DIR / f"{slug}__attempt_{attempt_number}.jsonl",
                    )

                rebuild_aggregate_artifacts()
                manifest = update_batch_manifest("running")
                log(
                    f"{dataset_id}/{example_id}: success={success}, "
                    f"release={row.get('release_status')}, "
                    f"writer={row.get('writer_mode')}, error={error}"
                )
                log(f"Batch completion: {manifest['completion']}")

                if PRINT_FINAL_OUTPUTS and generated_text:
                    display(
                        Markdown(
                            f"### {dataset_id} / {example_id}\n\n{generated_text}"
                        )
                    )
                if success:
                    case_succeeded = True
                    break
            except Exception as exc:
                append_jsonl(
                    ATTEMPT_LOG_PATH,
                    {
                        "event": "exception",
                        "timestamp": utc_now(),
                        "dataset_id": dataset_id,
                        "example_id": example_id,
                        "attempt": attempt_number,
                        "success": False,
                        "error": f"{type(exc).__name__}: {exc}",
                    },
                )
                log(
                    f"{dataset_id}/{example_id} attempt {attempt_number} raised "
                    f"{type(exc).__name__}: {exc}"
                )

        if not case_succeeded and not CONTINUE_AFTER_FAILURE:
            raise RuntimeError(f"Protected generation failed: {dataset_id}/{example_id}")

    final_frame = rebuild_aggregate_artifacts()
    final_manifest = update_batch_manifest()
    assert_frozen_state()
    log("=" * 80)
    log(f"Session complete. Batch status: {final_manifest['status']}")
    log(f"Completion: {final_manifest['completion']}")
    log(f"Execution report: {EXECUTION_REPORT_PATH}")
    return final_frame


protected_results = await run_protected_batch()

if not protected_results.empty:
    display(
        protected_results[
            [
                "dataset_id",
                "example_id",
                "execution_outcome",
                "final_generation_path",
                "release_status",
                "native_support_rate",
                "output_word_count",
                "verified_fact_count",
                "verified_insight_count",
                "fallback_event_count",
                "provider_reported_total_tokens",
                "elapsed_seconds",
            ]
        ]
    )




[01:28:23] ================================================================================
[01:28:23] Protected batch: 0/25 already complete; 25 scheduled this session.
[01:28:23] --------------------------------------------------------------------------------
[01:28:23] CASE 1/25 this session; protected position 1/25: e2e_nlg/e2e_nlg-test-1330
[01:28:43] e2e_nlg/e2e_nlg-test-1330 attempt 1/2: still running (20.0s elapsed)
[01:28:48] e2e_nlg/e2e_nlg-test-1330 attempt 1/2: completed after 25.8s
[01:28:48] e2e_nlg/e2e_nlg-test-1330: success=True, release=approved_with_warnings, writer=llm_writer, error=None
[01:28:48] Batch completion: {'successful': 1, 'failed': 0, 'not_started': 24}


### e2e_nlg / e2e_nlg-test-1330

The Vaults is a family-friendly pub in the city centre serving Italian food, with a customer rating of 3 out of 5, near Rainbow Vegetarian Café.


[01:28:48] --------------------------------------------------------------------------------
[01:28:48] CASE 2/25 this session; protected position 2/25: web_nlg/web_nlg_en-test-1209
[01:29:01] web_nlg/web_nlg_en-test-1209 attempt 1/2: completed after 12.7s
[01:29:01] web_nlg/web_nlg_en-test-1209: success=True, release=approved_with_warnings, writer=llm_writer, error=None
[01:29:01] Batch completion: {'successful': 2, 'failed': 0, 'not_started': 23}


### web_nlg / web_nlg_en-test-1209

Piotr Hallmann was born in Gdynia, Poland, which uses Central European Time and Central European Summer Time, and has a height of 175.26.


[01:29:01] --------------------------------------------------------------------------------
[01:29:01] CASE 3/25 this session; protected position 3/25: dart/dart-test-1791
[01:29:11] dart/dart-test-1791 attempt 1/2: completed after 9.8s
[01:29:11] dart/dart-test-1791: success=True, release=approved_with_warnings, writer=llm_writer, error=None
[01:29:11] Batch completion: {'successful': 3, 'failed': 0, 'not_started': 22}


...

### totto / totto-validation-912

In 2016, Tyrell Sutton played 7 games, with 74 carries for 412 yards and an average of 5.6 yards per carry.


[02:38:38] --------------------------------------------------------------------------------
[02:38:38] CASE 25/25 this session; protected position 25/25: sportsett_basketball/6127
[02:38:58] sportsett_basketball/6127 attempt 1/2: still running (20.0s elapsed)
[02:39:18] sportsett_basketball/6127 attempt 1/2: still running (40.0s elapsed)
[02:39:38] sportsett_basketball/6127 attempt 1/2: still running (60.0s elapsed)
[02:39:58] sportsett_basketball/6127 attempt 1/2: still running (80.0s elapsed)
[02:40:18] sportsett_basketball/6127 attempt 1/2: still running (100.0s elapsed)
[02:40:38] sportsett_basketball/6127 attempt 1/2: still running (120.1s elapsed)
[02:40:58] sportsett_basketball/6127 attempt 1/2: still running (140.1s elapsed)
[02:41:18] sportsett_basketball/6127 attempt 1/2: still running (160.1s elapsed)
[02:41:38] sportsett_basketball/6127 attempt 1/2: still running (180.1s elapsed)
[02:41:58] sportsett_basketball/6127 attempt 1/2: still running (200.1s elapsed)
[02:42:18] spo

### sportsett_basketball / 6127

The Washington Wizards defeated the Atlanta Hawks 114-98 at Capital One Arena on Wednesday, January 2, 2019, in a 2018-season matchup.
Washington entered with a 15-23 record in its 38th game, while Atlanta arrived at 11-26 in its 37th game.
Washington led at the end of every quarter, ahead 35-29 after the first, 64-53 at halftime, and 88-84 after three.
The Hawks' only quarter-winning margin was a 31-24 third period, while the Wizards closed with a 26-14 fourth quarter to seal the result.
Bradley Beal and Alex Len tied for game-high scoring honours with 24 points each; Jeff Green added 22 for the Wizards, and John Collins paced Atlanta with 21.
Len also led the game with 11 field goals made and 3 blocks, grabbed 11 rebounds and a game-high 6 offensive rebounds, while Thomas Bryant pulled down a game-high 15 rebounds (14 defensive) and Trae Young dished out a game-high 9 assists.
Tomáš Satoranský supported Washington with 11 rebounds and 7 assists, while Kevin Huerter and Jeremy Lin each dished out 5 assists for Atlanta.
Chasson Randle and Jeremy Lin each recorded a game-high 3 steals, and Jeff Green and John Collins each made a game-high 4 three-pointers.
The largest team-level gap in the supplied statistics came at the free-throw line, where the Wizards finished 17 of 23 and the Hawks 8 of 13.
Washington also made more field goals than Atlanta (43-40) despite fewer attempts (92-95), and held a 10-5 edge in steals while committing fewer turnovers (10-14).
Atlanta, meanwhile, controlled the glass, outrebounding Washington 50-48, and blocked more shots (6-2).
Washington held a 38-37 edge in defensive rebounds, while the Hawks led 13-10 on the offensive glass.
The Wizards also recorded a narrow 29-26 advantage in assists.
Both teams return to action on Friday, January 4, 2019, with Washington visiting the Miami Heat and Atlanta travelling to the Milwaukee Bucks.


[02:58:42] ================================================================================
[02:58:42] Session complete. Batch status: complete
[02:58:42] Completion: {'successful': 25, 'failed': 0, 'not_started': 0}
[02:58:42] Execution report: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_full_system/results/PROTECTED_HOLDOUT_EXECUTION_REPORT.md


,dataset_id,example_id,execution_outcome,final_generation_path,release_status,native_support_rate,output_word_count,verified_fact_count,verified_insight_count,fallback_event_count,provider_reported_total_tokens,elapsed_seconds
0,e2e_nlg,e2e_nlg-test-1330,success,normal_llm_writer,approved_with_warnings,1.0,26,1,0,1,7778,25.767118
1,web_nlg,web_nlg_en-test-1209,success,normal_llm_writer,approved_with_warnings,1.0,24,1,0,1,6806,12.709543
2,dart,dart-test-1791,success,normal_llm_writer,approved_with_warnings,1.0,20,1,0,1,6290,9.758940
3,totto,totto-validation-1828,success,deterministic_fallback,approved_with_warnings,1.0,25,1,0,2,8801,23.170308
4,sportsett_basketball,5130,success,deterministic_fallback,approved,1.0,299,41,5,1,514250,763.148925
5,e2e_nlg,e2e_nlg-test-209,success,normal_llm_writer,approved_with_warnings,1.0,10,1,0,1,6071,5.955769
6,web_nlg,web_nlg_en-test-1330,success,normal_llm_writer,approved_with_warnings,1.0,28,1,0,1,12413,9.308612
7,dart,dart-test-1805,success,normal_llm_writer,approved_with_warnings,1.0,14,1,0,1,6855,12.389781
8,totto,totto-validation-4467,success,normal_llm_writer,approved_with_warnings,1.0,18,1,0,1,14915,10.965698
9,sportsett_basketball,5372,success,auditor_repaired,approved,1.0,330,40,3,0,813172,1398.025812


## 6. Batch integrity checks

This cell is safe to rerun at any point. It reports missing, failed, or
incomplete cases without regenerating anything.



In [8]:
assert_frozen_state()
records = collect_final_records()
record_by_key = {
    (record.dataset_id, str(record.example_id)): record for record in records
}
integrity_rows = []
for item in selection_manifest["run_order"]:
    key = (item["dataset_id"], str(item["example_id"]))
    record = record_by_key.get(key)
    pipeline_path = locate_pipeline_result(record) if record else None
    integrity_rows.append(
        {
            "position": item["position"],
            "dataset_id": item["dataset_id"],
            "example_id": item["example_id"],
            "record_present": record is not None,
            "generation_success": bool(
                record and not record.error and record.generated_text.strip()
            ),
            "pipeline_result_present": bool(pipeline_path and pipeline_path.exists()),
            "reference_masked_in_record": bool(
                record and record.references == [HELD_OUT_REFERENCE_SENTINEL]
            ),
            "model_input_isolation_hash_match": bool(
                record
                and pipeline_path
                and extract_summary(record)[0].get(
                    "model_input_matches_reference_isolation_audit"
                )
            ),
            "error": record.error if record else None,
        }
    )

integrity_frame = pd.DataFrame(integrity_rows)
display(integrity_frame)
print("Successful:", int(integrity_frame["generation_success"].sum()), "/", expected_total)
print("All references masked:", bool(integrity_frame["reference_masked_in_record"].all()))
print("Implementation fingerprint unchanged: PASS")




,position,dataset_id,example_id,record_present,generation_success,pipeline_result_present,reference_masked_in_record,model_input_isolation_hash_match,error
0,1,e2e_nlg,e2e_nlg-test-1330,True,True,True,True,True,None
1,2,web_nlg,web_nlg_en-test-1209,True,True,True,True,True,None
2,3,dart,dart-test-1791,True,True,True,True,True,None
3,4,totto,totto-validation-1828,True,True,True,True,True,None
4,5,sportsett_basketball,5130,True,True,True,True,True,None
5,6,e2e_nlg,e2e_nlg-test-209,True,True,True,True,True,None
6,7,web_nlg,web_nlg_en-test-1330,True,True,True,True,True,None
7,8,dart,dart-test-1805,True,True,True,True,True,None
8,9,totto,totto-validation-4467,True,True,True,True,True,None
9,10,sportsett_basketball,5372,True,True,True,True,True,None


Successful: 25 / 25
All references masked: True
Implementation fingerprint unchanged: PASS


## 7. Optional post-generation reference and source metrics

This phase is blocked until all 25 protected generations are successful.
Only here are the true references joined back into a separate copy of the
GenerationRecords. The sealed generation file and all workflow artifacts are
left untouched.



In [9]:
def unseal_generation_records_for_metrics():
    records = collect_final_records()
    successful = [record for record in records if not record.error and record.generated_text]
    if len(successful) != expected_total:
        raise RuntimeError(
            f"Cannot unseal references: {len(successful)}/{expected_total} "
            "protected generations are successful."
        )
    assert_frozen_state()
    true_lookup = {
        (example.dataset_id, str(example.example_id)): example
        for example in selected_examples
    }
    unsealed = [
        record.model_copy(
            update={
                "references": true_lookup[
                    (record.dataset_id, str(record.example_id))
                ].references
            }
        )
        for record in successful
    ]
    write_jsonl_atomic(UNSEALED_GENERATIONS_PATH, unsealed)
    manifest = read_json(BATCH_MANIFEST_PATH, {})
    if not manifest.get("true_references_unsealed_at"):
        manifest["true_references_unsealed_at"] = utc_now()
        manifest["true_references_unsealed_for"] = "post-generation evaluation only"
        manifest["unsealed_generations_path"] = relative_path(UNSEALED_GENERATIONS_PATH)
        write_json_atomic(BATCH_MANIFEST_PATH, manifest)
    return unsealed


def build_metric_config(enabled_metrics, context, filename):
    payload = copy.deepcopy(read_json(PATHS["metric_config"]))
    payload["experiment_id"] = f"{EXPERIMENT_ID}_{context}"
    payload["prepared_examples_path"] = relative_path(OPERATIONAL_EXAMPLES_PATH)
    payload["generations_path"] = relative_path(UNSEALED_GENERATIONS_PATH)
    payload["result_directory"] = relative_path(RESULT_DIR)
    payload["baseline_variant"] = "full_system"
    payload["reference_metrics"]["enabled_metrics"] = enabled_metrics
    payload["reference_metrics"]["external_factuality_context"] = context
    payload["deepeval"]["enabled"] = False
    path = CONFIG_DIR / filename
    write_json_atomic(path, payload)
    return path


reference_scores = pd.DataFrame()
source_scores = pd.DataFrame()

if RUN_POST_GENERATION_REFERENCE_METRICS or RUN_POST_GENERATION_SOURCE_METRICS:
    unseal_generation_records_for_metrics()

if RUN_POST_GENERATION_REFERENCE_METRICS:
    reference_config_path = build_metric_config(
        [
            "bleu",
            "chrf",
            "ter",
            "rouge1",
            "rouge2",
            "rougeL",
            "rougeLsum",
            "meteor",
            "bertscore",
            "parent",
        ],
        "references",
        "metrics_protected_reference.json",
    )
    log("Starting protected post-generation reference metrics")
    reference_scores = score_reference_metrics_for_notebook(
        PROJECT_DIR,
        generations_path=UNSEALED_GENERATIONS_PATH,
        metric_config_path=reference_config_path,
        output_path=RESULT_DIR / "protected_reference_metrics.jsonl",
        include_ineligible=True,
    )
    display(
        reference_scores.pivot_table(
            index="metric_name",
            values="score",
            aggfunc="mean",
        ).sort_index()
    )

if RUN_POST_GENERATION_SOURCE_METRICS:
    source_config_path = build_metric_config(
        ["hhem", "alignscore"],
        "source_text",
        "metrics_protected_source_grounded.json",
    )
    log("Starting protected post-generation source-grounded metrics")
    source_scores = score_reference_metrics_for_notebook(
        PROJECT_DIR,
        generations_path=UNSEALED_GENERATIONS_PATH,
        metric_config_path=source_config_path,
        output_path=RESULT_DIR / "protected_source_grounded_metrics.jsonl",
        include_ineligible=True,
    )
    display(
        source_scores.pivot_table(
            index="metric_name",
            values="score",
            aggfunc="mean",
        ).sort_index()
    )

print("Protected generation artifacts:", ARTIFACT_DIR)
print("Batch manifest:", BATCH_MANIFEST_PATH)
print("Execution report:", EXECUTION_REPORT_PATH)
print("Exact outputs and summaries:", SUMMARY_JSONL_PATH)


Protected generation artifacts: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_full_system
Batch manifest: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_full_system/results/protected_batch_manifest.json
Execution report: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_full_system/results/PROTECTED_HOLDOUT_EXECUTION_REPORT.md
Exact outputs and summaries: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_full_system/results/protected_generation_summary.jsonl


In [10]:
import asyncio
import hashlib
import json
import os
import random
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from openai import OpenAI

from table2text.evaluation import (
    annotate_with_openai_judge_for_notebook,
    default_paths,
    load_project_env,
)
from table2text.evaluation.datasets import read_examples
from table2text.evaluation.llm_judge_annotations import (
    OpenAIJudgeAnnotationConfig,
    build_annotation_judge_input,
)
from table2text.evaluation.models import (
    GenerationBackend,
    GenerationRecord,
    LLMJudgeAnnotationRecord,
)

# ============================================================
# Configuration
# ============================================================

project_dir = Path(
    "/Users/realgobs/Documents/MScproject/table2text_pydanticai"
)
paths = default_paths(project_dir)
load_project_env(project_dir)

RUN_BASELINE = True
RUN_GPT56_JUDGE = True

# Set to an integer for a small resumable batch, or None for all.
MAX_BASELINE_CASES_THIS_SESSION = None
MAX_JUDGE_OUTPUTS_THIS_SESSION = None

HEARTBEAT_SECONDS = 20

GENERIC_REQUEST = (
    "Understand the supplied data and report its strongest supported findings."
)

BASELINE_MODEL = "deepseek-v4-flash"
BASELINE_TEMPERATURE = 0.2
BASELINE_MAX_SOURCE_CHARACTERS = 100_000
BASELINE_MAX_OUTPUT_TOKENS = 3_000
BASELINE_SEED = 42

JUDGE_MODEL = "gpt-5.6-sol"
JUDGE_REASONING_EFFORT = "high"
JUDGE_MAX_SOURCE_CHARACTERS = 50_000
JUDGE_MAX_OUTPUT_TOKENS = 2_500

REFERENCE_SENTINEL = (
    "<HELD_OUT_REFERENCE_NOT_AVAILABLE_DURING_GENERATION>"
)

CONDITION_LABELS = {
    "full_system": "Full System",
    "baseline": "Baseline",
}

# ============================================================
# Paths
# ============================================================

full_root = (
    project_dir / "evaluation/protected_holdout_full_system"
)
output_root = (
    project_dir / "evaluation/protected_holdout_baseline"
)

selection_path = (
    full_root / "prepared/protected_selection_manifest.json"
)
operational_examples_path = (
    full_root / "prepared/protected_operational_examples.jsonl"
)
full_generations_path = (
    full_root
    / "generations/protected_full_system_generations_sealed.jsonl"
)

baseline_shard_dir = output_root / "generations/shards"
baseline_generations_path = (
    output_root / "generations/baseline_generations_sealed.jsonl"
)
paired_generations_path = (
    output_root
    / "comparison/full_system_and_baseline_sealed.jsonl"
)

judge_input_dir = output_root / "gpt56_judge/inputs"
judge_shard_dir = output_root / "gpt56_judge/shards"
judge_annotations_path = (
    output_root
    / "gpt56_judge/results/gpt56_structured_annotations.jsonl"
)
judge_summary_path = (
    output_root
    / "gpt56_judge/results/gpt56_annotation_summary.csv"
)
judge_categories_path = (
    output_root
    / "gpt56_judge/results/gpt56_category_counts.csv"
)
progress_log_path = output_root / "results/progress.log"

for directory in [
    baseline_shard_dir,
    paired_generations_path.parent,
    judge_input_dir,
    judge_shard_dir,
    judge_annotations_path.parent,
    progress_log_path.parent,
]:
    directory.mkdir(parents=True, exist_ok=True)

# ============================================================
# Helpers
# ============================================================

def log(message):
    line = f"[{datetime.now().strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with progress_log_path.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def safe_name(dataset_id, example_id):
    value = f"{dataset_id}__{example_id}"
    return "".join(
        c if c.isalnum() or c in "-_" else "_"
        for c in value
    )


def write_jsonl(path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")

    with temporary.open("w", encoding="utf-8") as handle:
        for record in records:
            payload = (
                record.model_dump(mode="json")
                if hasattr(record, "model_dump")
                else record
            )
            handle.write(
                json.dumps(payload, ensure_ascii=False) + "\n"
            )

    temporary.replace(path)


def read_generations(path):
    if not path.exists():
        return []

    return [
        GenerationRecord.model_validate_json(line)
        for line in path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]


def read_annotations(path):
    if not path.exists():
        return []

    return [
        LLMJudgeAnnotationRecord.model_validate_json(line)
        for line in path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]


async def run_with_heartbeat(function, *args, label):
    task = asyncio.create_task(
        asyncio.to_thread(function, *args)
    )
    started = time.perf_counter()

    while True:
        try:
            result = await asyncio.wait_for(
                asyncio.shield(task),
                timeout=HEARTBEAT_SECONDS,
            )
            elapsed = time.perf_counter() - started
            log(f"Finished {label} after {elapsed:.1f}s")
            return result
        except asyncio.TimeoutError:
            elapsed = time.perf_counter() - started
            log(f"Still running {label}; elapsed={elapsed:.1f}s")


def source_for_baseline(example):
    source = example.source_text.strip()

    if not source:
        source = json.dumps(
            example.source_payload,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )

    if len(source) <= BASELINE_MAX_SOURCE_CHARACTERS:
        return source, False

    return (
        source[:BASELINE_MAX_SOURCE_CHARACTERS]
        + "\n\n[Source truncated by Baseline configuration.]",
        True,
    )


def build_baseline_messages(example):
    source, _ = source_for_baseline(example)

    return [
        {
            "role": "system",
            "content": (
                "You are a data-to-text baseline. Generate a useful "
                "factual answer directly from the supplied source data "
                "and generic request. Use only the supplied source. "
                "Do not use outside knowledge or invent entities, "
                "numbers, chronology, causal claims, or background. "
                "Return only the final answer."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Request:\n{GENERIC_REQUEST}\n\n"
                f"Source data:\n{source}\n\n"
                "Write the final answer only."
            ),
        },
    ]


def run_baseline(example):
    started = time.perf_counter()
    generation_id = (
        f"{example.dataset_id}__{example.example_id}"
        f"__baseline__r0__s{BASELINE_SEED}"
    )

    try:
        api_key = os.getenv("DEEPSEEK_API_KEY", "").strip()
        if not api_key:
            raise RuntimeError(
                "DEEPSEEK_API_KEY is not set in .env."
            )

        client = OpenAI(
            api_key=api_key,
            base_url=os.getenv(
                "DEEPSEEK_BASE_URL",
                "https://api.deepseek.com",
            ),
        )

        response = client.chat.completions.create(
            model=BASELINE_MODEL,
            messages=build_baseline_messages(example),
            temperature=BASELINE_TEMPERATURE,
            max_tokens=BASELINE_MAX_OUTPUT_TOKENS,
        )

        generated_text = (
            response.choices[0].message.content or ""
        ).strip()

        usage = getattr(response, "usage", None)
        _, source_truncated = source_for_baseline(example)

        return GenerationRecord(
            generation_id=generation_id,
            dataset_id=example.dataset_id,
            example_id=example.example_id,
            variant_id="baseline",
            repetition=0,
            seed=BASELINE_SEED,
            task_family=example.task_family,
            output_mode=example.output_mode,
            language=example.language,
            source_text=example.source_text,
            references=example.references,
            parent_table=example.parent_table,
            request=GENERIC_REQUEST,
            generated_text=generated_text,
            backend=GenerationBackend.CALLABLE,
            primary_evaluation_eligible=True,
            elapsed_seconds=time.perf_counter() - started,
            input_tokens=(
                getattr(usage, "prompt_tokens", None)
                if usage else None
            ),
            output_tokens=(
                getattr(usage, "completion_tokens", None)
                if usage else None
            ),
            total_tokens=(
                getattr(usage, "total_tokens", None)
                if usage else None
            ),
            metadata={
                "condition": "Baseline",
                "method": "single_direct_model_call",
                "provider": "deepseek",
                "model": BASELINE_MODEL,
                "temperature": BASELINE_TEMPERATURE,
                "generic_request": GENERIC_REQUEST,
                "task_metadata_supplied": False,
                "task_specific_request_supplied": False,
                "reference_supplied": False,
                "source_truncated": source_truncated,
            },
        )

    except Exception as exc:
        return GenerationRecord(
            generation_id=generation_id,
            dataset_id=example.dataset_id,
            example_id=example.example_id,
            variant_id="baseline",
            repetition=0,
            seed=BASELINE_SEED,
            task_family=example.task_family,
            output_mode=example.output_mode,
            language=example.language,
            source_text=example.source_text,
            references=example.references,
            parent_table=example.parent_table,
            request=GENERIC_REQUEST,
            generated_text="",
            backend=GenerationBackend.CALLABLE,
            primary_evaluation_eligible=False,
            primary_evaluation_reason="generation_error",
            elapsed_seconds=time.perf_counter() - started,
            metadata={
                "condition": "Baseline",
                "method": "single_direct_model_call",
                "model": BASELINE_MODEL,
            },
            error=f"{type(exc).__name__}: {exc}",
        )


def baseline_shard(example):
    return (
        baseline_shard_dir
        / f"{safe_name(example.dataset_id, example.example_id)}.jsonl"
    )


def completed_baseline(example):
    records = read_generations(baseline_shard(example))
    if len(records) != 1:
        return None

    record = records[0]
    if record.error is None and record.generated_text.strip():
        return record

    return None


# ============================================================
# Load and validate the exact protected selection
# ============================================================

selection = json.loads(selection_path.read_text(encoding="utf-8"))
selection_order = [
    (str(item["dataset_id"]), str(item["example_id"]))
    for item in selection["run_order"]
]

operational_examples = read_examples(operational_examples_path)
operational_map = {
    (example.dataset_id, str(example.example_id)): example
    for example in operational_examples
}

all_examples = read_examples(paths["prepared_examples"])
true_example_map = {
    (example.dataset_id, str(example.example_id)): example
    for example in all_examples
    if (example.dataset_id, str(example.example_id))
    in set(selection_order)
}

full_records = read_generations(full_generations_path)
full_map = {
    (record.dataset_id, str(record.example_id)): record
    for record in full_records
}

assert len(selection_order) == 25
assert len(set(selection_order)) == 25
assert set(operational_map) == set(selection_order)
assert set(true_example_map) == set(selection_order)
assert set(full_map) == set(selection_order)
assert all(
    example.references == [REFERENCE_SENTINEL]
    for example in operational_examples
)
assert all(
    record.references == [REFERENCE_SENTINEL]
    for record in full_records
)
assert all(
    record.error is None
    and record.generated_text.strip()
    and record.primary_evaluation_eligible is True
    for record in full_records
)

# Replace every task-specific generation request with one generic request.
baseline_examples = [
    operational_map[key].model_copy(
        update={"request": GENERIC_REQUEST}
    )
    for key in selection_order
]

# Prompt-isolation audit.
for example in baseline_examples:
    messages = build_baseline_messages(example)
    prompt = "\n\n".join(m["content"] for m in messages)

    assert GENERIC_REQUEST in prompt
    assert REFERENCE_SENTINEL not in prompt
    assert "Task type:" not in prompt
    assert "Expected form:" not in prompt
    assert "Language:" not in prompt

    original_request = operational_map[
        (example.dataset_id, str(example.example_id))
    ].request

    if original_request != GENERIC_REQUEST:
        assert original_request not in prompt

log("Protected selection verified: exact same 25 examples.")
log("Baseline prompt isolation verified for all 25 examples.")

# ============================================================
# Generate the Baseline
# ============================================================

pending = [
    example
    for example in baseline_examples
    if completed_baseline(example) is None
]

if MAX_BASELINE_CASES_THIS_SESSION is not None:
    pending = pending[:MAX_BASELINE_CASES_THIS_SESSION]

completed_before = 25 - len([
    example
    for example in baseline_examples
    if completed_baseline(example) is None
])

log(f"Baseline status before session: {completed_before}/25 complete.")

if RUN_BASELINE:
    for index, example in enumerate(pending, start=1):
        label = f"{example.dataset_id}/{example.example_id}"

        log(f"Baseline {index}/{len(pending)}: starting {label}")

        record = await run_with_heartbeat(
            run_baseline,
            example,
            label=f"Baseline {label}",
        )

        write_jsonl(baseline_shard(example), [record])

        if record.error:
            log(f"Baseline failed for {label}: {record.error}")
            continue

        log(
            f"Baseline completed {label}; "
            f"seconds={record.elapsed_seconds:.1f}; "
            f"tokens={record.total_tokens}"
        )

        display(
            Markdown(
                f"### Baseline: {label}\n\n"
                f"{record.generated_text}"
            )
        )

# Consolidate successful Baseline records.
baseline_map = {
    (example.dataset_id, str(example.example_id)):
    completed_baseline(example)
    for example in baseline_examples
}

baseline_complete = all(
    record is not None for record in baseline_map.values()
)

baseline_records = [
    baseline_map[key]
    for key in selection_order
    if baseline_map[key] is not None
]

log(f"Baseline consolidated status: {len(baseline_records)}/25.")

if baseline_complete:
    write_jsonl(baseline_generations_path, baseline_records)

    paired_records = []
    for key in selection_order:
        paired_records.extend([
            full_map[key],
            baseline_map[key],
        ])

    assert len(paired_records) == 50
    assert all(
        record.references == [REFERENCE_SENTINEL]
        for record in paired_records
    )

    write_jsonl(paired_generations_path, paired_records)
    log("All 25 Baseline outputs sealed and paired with Full System.")
else:
    paired_records = []
    log(
        "Baseline is incomplete. Rerun this cell to resume before "
        "starting GPT-5.6 Sol."
    )

# ============================================================
# Prepare blinded GPT-5.6 Sol judge records
# ============================================================

def judge_shard(record):
    filename = (
        f"{safe_name(record.dataset_id, record.example_id)}"
        f"__{record.variant_id}.jsonl"
    )
    return judge_shard_dir / filename


def completed_annotation(record):
    annotations = read_annotations(judge_shard(record))

    if len(annotations) != 1:
        return None

    annotation = annotations[0]
    status = getattr(annotation.status, "value", annotation.status)

    if status == "scored" and annotation.error is None:
        return annotation

    return None


def run_judge(record):
    stem = (
        f"{safe_name(record.dataset_id, record.example_id)}"
        f"__{record.variant_id}"
    )
    input_path = judge_input_dir / f"{stem}.jsonl"
    output_path = judge_shard(record)

    write_jsonl(input_path, [record])

    return annotate_with_openai_judge_for_notebook(
        project_dir,
        generations_path=input_path,
        output_path=output_path,
        judge_model=JUDGE_MODEL,
        judge_repetitions=1,
        reasoning_effort=JUDGE_REASONING_EFFORT,
        max_source_characters=JUDGE_MAX_SOURCE_CHARACTERS,
        max_output_tokens=JUDGE_MAX_OUTPUT_TOKENS,
        include_references=False,
        include_system_identity=False,
        include_metric_scores=False,
        resume=False,
    )


if RUN_GPT56_JUDGE:
    if not baseline_complete:
        raise RuntimeError(
            "The Baseline must reach 25/25 before judging."
        )

    if not (
        os.getenv("OPENAI_API_KEY")
        or os.getenv("T2T_OPENAI_API_KEY")
    ):
        raise RuntimeError(
            "OPENAI_API_KEY is not set in .env."
        )

    judge_config = OpenAIJudgeAnnotationConfig(
        judge_model=JUDGE_MODEL,
        judge_repetitions=1,
        reasoning_effort=JUDGE_REASONING_EFFORT,
        max_source_characters=JUDGE_MAX_SOURCE_CHARACTERS,
        max_output_tokens=JUDGE_MAX_OUTPUT_TOKENS,
        include_references=False,
        include_system_identity=False,
        include_metric_scores=False,
    )

    # Restore the common benchmark task only for evaluation.
    # References remain withheld.
    judge_records = []

    for record in paired_records:
        key = (record.dataset_id, str(record.example_id))
        benchmark_example = true_example_map[key]

        judge_record = record.model_copy(
            update={
                "request": benchmark_example.request,
                "references": [REFERENCE_SENTINEL],
                "primary_evaluation_eligible": True,
            }
        )

        judge_prompt = build_annotation_judge_input(
            judge_record,
            judge_config,
        )

        assert "HUMAN REFERENCES" not in judge_prompt
        assert "SYSTEM IDENTITY" not in judge_prompt
        assert "SYSTEM METADATA" not in judge_prompt
        assert REFERENCE_SENTINEL not in judge_prompt
        assert benchmark_example.request in judge_prompt

        judge_records.append(judge_record)

    assert len(judge_records) == 50

    # Reproducible order, while avoiding paired presentation.
    random.Random(56042).shuffle(judge_records)

    pending_judgements = [
        record
        for record in judge_records
        if completed_annotation(record) is None
    ]

    if MAX_JUDGE_OUTPUTS_THIS_SESSION is not None:
        pending_judgements = pending_judgements[
            :MAX_JUDGE_OUTPUTS_THIS_SESSION
        ]

    completed_before = 50 - len([
        record
        for record in judge_records
        if completed_annotation(record) is None
    ])

    log(
        f"GPT-5.6 Sol status before session: "
        f"{completed_before}/50 complete."
    )

    for index, record in enumerate(
        pending_judgements,
        start=1,
    ):
        condition = CONDITION_LABELS[record.variant_id]
        label = (
            f"{record.dataset_id}/{record.example_id} "
            f"({condition})"
        )

        log(
            f"GPT-5.6 Sol {index}/{len(pending_judgements)}: "
            f"starting {label}"
        )

        frame = await run_with_heartbeat(
            run_judge,
            record,
            label=f"GPT-5.6 Sol {label}",
        )

        annotation = completed_annotation(record)

        if annotation is None:
            error = (
                frame.iloc[0].get("error")
                if not frame.empty
                else "No annotation returned"
            )
            log(f"Judge failed for {label}: {error}")
            continue

        log(
            f"Judge completed {label}; "
            f"errors={annotation.error_count}; "
            f"tokens={annotation.total_tokens}"
        )

        if annotation.errors:
            display(pd.DataFrame([
                error.model_dump(mode="json")
                for error in annotation.errors
            ]))
        else:
            print("No errors reported.")

    # Consolidate successful annotations.
    annotation_map = {}

    for record in judge_records:
        annotation = completed_annotation(record)
        if annotation is not None:
            annotation_map[record.generation_id] = annotation

    annotations = [
        annotation_map[record.generation_id]
        for record in judge_records
        if record.generation_id in annotation_map
    ]

    write_jsonl(judge_annotations_path, annotations)

    summary_rows = []
    category_rows = []

    categories = [
        "NAME",
        "NUMBER",
        "WORD",
        "CONTEXT",
        "NOT CHECKABLE",
        "OTHER",
        "OMISSION",
        "TASK/FORMAT",
    ]

    for annotation in annotations:
        condition = CONDITION_LABELS[annotation.variant_id]
        counts = Counter(
            error.category for error in annotation.errors
        )

        summary_rows.append({
            "dataset_id": annotation.dataset_id,
            "example_id": annotation.example_id,
            "condition": condition,
            "judge_model": annotation.judge_model,
            "error_count": annotation.error_count,
            "duration_seconds": annotation.duration_seconds,
            "input_tokens": annotation.input_tokens,
            "output_tokens": annotation.output_tokens,
            "total_tokens": annotation.total_tokens,
        })

        for category in categories:
            category_rows.append({
                "dataset_id": annotation.dataset_id,
                "example_id": annotation.example_id,
                "condition": condition,
                "category": category,
                "count": counts.get(category, 0),
            })

    summary = pd.DataFrame(summary_rows)
    category_counts = pd.DataFrame(category_rows)

    summary.to_csv(judge_summary_path, index=False)
    category_counts.to_csv(
        judge_categories_path,
        index=False,
    )

    log(
        f"GPT-5.6 Sol consolidated status: "
        f"{len(annotations)}/50 scored."
    )

    if not summary.empty:
        print("\nOverall structured-annotation results")
        display(
            summary.groupby(
                "condition",
                as_index=False,
            ).agg(
                outputs=("example_id", "count"),
                total_errors=("error_count", "sum"),
                mean_errors=("error_count", "mean"),
                outputs_without_errors=(
                    "error_count",
                    lambda values: int((values == 0).sum()),
                ),
                total_judge_tokens=("total_tokens", "sum"),
            )
        )

        print("\nResults by dataset")
        display(
            summary.groupby(
                ["dataset_id", "condition"],
                as_index=False,
            ).agg(
                outputs=("example_id", "count"),
                total_errors=("error_count", "sum"),
                mean_errors=("error_count", "mean"),
            )
        )

        print("\nError categories")
        display(
            category_counts.groupby(
                ["condition", "category"],
                as_index=False,
            )["count"]
            .sum()
            .pivot(
                index="category",
                columns="condition",
                values="count",
            )
            .fillna(0)
            .astype(int)
            .reset_index()
        )

print("\nArtifacts")
print("Baseline generations:", baseline_generations_path)
print("Paired sealed outputs:", paired_generations_path)
print("GPT-5.6 Sol annotations:", judge_annotations_path)
print("Judge summary:", judge_summary_path)
print("Judge category counts:", judge_categories_path)
print("Progress log:", progress_log_path)

[03:20:29] Protected selection verified: exact same 25 examples.
[03:20:29] Baseline prompt isolation verified for all 25 examples.
[03:20:29] Baseline status before session: 0/25 complete.
[03:20:29] Baseline 1/25: starting e2e_nlg/e2e_nlg-test-1330
[03:20:30] Finished Baseline e2e_nlg/e2e_nlg-test-1330 after 1.7s
[03:20:30] Baseline completed e2e_nlg/e2e_nlg-test-1330; seconds=1.7; tokens=297


### Baseline: e2e_nlg/e2e_nlg-test-1330

The Vaults is a city-centre pub serving Italian food, with a customer rating of 3 out of 5. It is family friendly and located near Rainbow Vegetarian Café.

[03:20:30] Baseline 2/25: starting web_nlg/web_nlg_en-test-1209
[03:20:32] Finished Baseline web_nlg/web_nlg_en-test-1209 after 1.8s
[03:20:32] Baseline completed web_nlg/web_nlg_en-test-1209; seconds=1.8; tokens=360


### Baseline: web_nlg/web_nlg_en-test-1209

Piotr Hallmann was born in Gdynia, Poland. Gdynia uses Central European Time and Central European Summer Time. Piotr Hallmann's height is listed as 175.26.

[03:20:32] Baseline 3/25: starting dart/dart-test-1791
[03:20:34] Finished Baseline dart/dart-test-1791 after 1.7s
[03:20:34] Baseline completed dart/dart-test-1791; seconds=1.7; tokens=298


...


Results by dataset


,dataset_id,condition,outputs,total_errors,mean_errors
0,dart,Baseline,5,0,0.0
1,dart,Full System,5,0,0.0
2,e2e_nlg,Baseline,5,0,0.0
3,e2e_nlg,Full System,5,0,0.0
4,sportsett_basketball,Baseline,5,7,1.4
5,sportsett_basketball,Full System,5,13,2.6
6,totto,Baseline,5,8,1.6
7,totto,Full System,5,3,0.6
8,web_nlg,Baseline,5,1,0.2
9,web_nlg,Full System,5,0,0.0



Error categories


condition,category,Baseline,Full System
0,CONTEXT,2,6
1,NAME,0,0
2,NOT CHECKABLE,0,0
3,NUMBER,1,0
4,OMISSION,1,2
5,OTHER,0,0
6,TASK/FORMAT,11,6
7,WORD,1,2



Artifacts
Baseline generations: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/generations/baseline_generations_sealed.jsonl
Paired sealed outputs: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/comparison/full_system_and_baseline_sealed.jsonl
GPT-5.6 Sol annotations: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/gpt56_judge/results/gpt56_structured_annotations.jsonl
Judge summary: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/gpt56_judge/results/gpt56_annotation_summary.csv
Judge category counts: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/gpt56_judge/results/gpt56_category_counts.csv
Progress log: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/results/progress.log


In [11]:
import asyncio
import copy
import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

from table2text.evaluation import (
    default_paths,
    score_reference_metrics_for_notebook,
)
from table2text.evaluation.datasets import read_examples
from table2text.evaluation.models import GenerationRecord

# ============================================================
# Configuration
# ============================================================

project_dir = Path(
    "/Users/realgobs/Documents/MScproject/table2text_pydanticai"
)
paths = default_paths(project_dir)

RUN_REFERENCE_METRICS = True
RUN_SOURCE_GROUNDED_METRICS = True

HEARTBEAT_SECONDS = 20

CONDITION_LABELS = {
    "full_system": "Full System",
    "baseline": "Baseline",
}

REFERENCE_METRICS = [
    "bleu",
    "chrf",
    "ter",
    "rouge1",
    "rouge2",
    "rougeL",
    "rougeLsum",
    "meteor",
    "bertscore",
    "parent",
]

SOURCE_GROUNDED_METRICS = [
    "hhem",
    "alignscore",
]

# ============================================================
# Paths
# ============================================================

evaluation_root = (
    project_dir / "evaluation/protected_holdout_baseline"
)

sealed_generations_path = (
    evaluation_root
    / "comparison/full_system_and_baseline_sealed.jsonl"
)

metrics_generations_path = (
    evaluation_root
    / "comparison/full_system_and_baseline_for_metrics.jsonl"
)

config_dir = evaluation_root / "config"
results_dir = evaluation_root / "results"

config_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

reference_config_path = (
    config_dir / "metrics_reference_alignment.json"
)
source_config_path = (
    config_dir / "metrics_source_grounded.json"
)

reference_results_path = (
    results_dir / "reference_alignment_metrics.jsonl"
)
source_results_path = (
    results_dir / "source_grounded_metrics.jsonl"
)

overall_table_path = (
    results_dir / "automatic_metrics_overall.csv"
)
dataset_table_path = (
    results_dir / "automatic_metrics_by_dataset.csv"
)

# ============================================================
# Helpers
# ============================================================

def log(message):
    print(
        f"[{datetime.now().strftime('%H:%M:%S')}] {message}",
        flush=True,
    )


def read_generations(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Generation file does not exist:\n{path}\n\n"
            "Finish the Baseline generation cell first."
        )

    return [
        GenerationRecord.model_validate_json(line)
        for line in path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]


def write_jsonl(path, records):
    temporary = path.with_suffix(path.suffix + ".tmp")

    with temporary.open("w", encoding="utf-8") as handle:
        for record in records:
            payload = (
                record.model_dump(mode="json")
                if hasattr(record, "model_dump")
                else record
            )
            handle.write(
                json.dumps(payload, ensure_ascii=False) + "\n"
            )

    temporary.replace(path)


def write_json(path, payload):
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )


async def run_with_heartbeat(function, label):
    task = asyncio.create_task(asyncio.to_thread(function))
    started = time.perf_counter()

    while True:
        try:
            result = await asyncio.wait_for(
                asyncio.shield(task),
                timeout=HEARTBEAT_SECONDS,
            )
            elapsed = time.perf_counter() - started
            log(f"Finished {label} after {elapsed:.1f}s")
            return result
        except asyncio.TimeoutError:
            elapsed = time.perf_counter() - started
            log(f"Still running {label}; elapsed={elapsed:.1f}s")


def build_metric_config(
    enabled_metrics,
    factuality_context,
    output_path,
):
    payload = copy.deepcopy(
        json.loads(
            paths["metric_config"].read_text(encoding="utf-8")
        )
    )

    # These top-level fields are required by ExperimentConfig.
    payload["experiment_id"] = (
        "protected_holdout_full_system_vs_baseline"
    )
    payload["prepared_examples_path"] = str(
        paths["prepared_examples"]
    )
    payload["generations_path"] = str(
        metrics_generations_path
    )
    payload["result_directory"] = str(results_dir)
    payload["baseline_variant"] = "baseline"

    payload["reference_metrics"]["enabled_metrics"] = (
        enabled_metrics
    )
    payload["reference_metrics"][
        "external_factuality_context"
    ] = factuality_context

    # GPT-5.6 structured annotation is run separately.
    payload["deepeval"]["enabled"] = False

    write_json(output_path, payload)
    return output_path


def scored_only(frame):
    if frame.empty:
        return frame

    result = frame[frame["status"] == "scored"].copy()
    result["condition"] = result["variant_id"].map(
        CONDITION_LABELS
    )
    return result


# ============================================================
# Restore true references after generation
# ============================================================

sealed_records = read_generations(sealed_generations_path)

counts = pd.Series(
    [record.variant_id for record in sealed_records]
).value_counts()

assert counts.get("full_system", 0) == 25, counts
assert counts.get("baseline", 0) == 25, counts

all_examples = read_examples(paths["prepared_examples"])

example_map = {
    (example.dataset_id, str(example.example_id)): example
    for example in all_examples
}

metrics_records = []

for record in sealed_records:
    key = (record.dataset_id, str(record.example_id))

    if key not in example_map:
        raise KeyError(
            f"Could not find the prepared example for {key}."
        )

    example = example_map[key]

    metrics_records.append(
        record.model_copy(
            update={
                # References are added only to this metrics copy.
                "references": example.references,
                "request": example.request,
                "primary_evaluation_eligible": True,
            }
        )
    )

assert len(metrics_records) == 50
assert all(
    record.references
    and record.references != [
        "<HELD_OUT_REFERENCE_NOT_AVAILABLE_DURING_GENERATION>"
    ]
    for record in metrics_records
)

write_jsonl(metrics_generations_path, metrics_records)

log("Prepared 50 post-generation metric records.")
log("True references were added only to the metrics copy.")

# ============================================================
# Reference-alignment metrics
# ============================================================

all_score_frames = []

if RUN_REFERENCE_METRICS:
    build_metric_config(
        REFERENCE_METRICS,
        "references",
        reference_config_path,
    )

    def run_reference_metrics():
        return score_reference_metrics_for_notebook(
            project_dir,
            generations_path=metrics_generations_path,
            metric_config_path=reference_config_path,
            output_path=reference_results_path,
            include_ineligible=True,
        )

    log("Starting reference-alignment metrics...")

    reference_scores = await run_with_heartbeat(
        run_reference_metrics,
        "reference-alignment metrics",
    )

    reference_scores = scored_only(reference_scores)
    all_score_frames.append(reference_scores)

    print("\nReference-alignment metrics")
    display(
        reference_scores.groupby(
            ["metric_name", "condition"],
            as_index=False,
        )["score"]
        .mean()
        .pivot(
            index="metric_name",
            columns="condition",
            values="score",
        )
        .reset_index()
    )

# ============================================================
# Source-grounded metrics
# ============================================================

if RUN_SOURCE_GROUNDED_METRICS:
    build_metric_config(
        SOURCE_GROUNDED_METRICS,
        "source_text",
        source_config_path,
    )

    def run_source_metrics():
        return score_reference_metrics_for_notebook(
            project_dir,
            generations_path=metrics_generations_path,
            metric_config_path=source_config_path,
            output_path=source_results_path,
            include_ineligible=True,
        )

    log("Starting source-grounded HHEM and AlignScore...")

    source_scores = await run_with_heartbeat(
        run_source_metrics,
        "source-grounded metrics",
    )

    source_scores = scored_only(source_scores)
    all_score_frames.append(source_scores)

    print("\nSource-grounded metrics")
    display(
        source_scores.groupby(
            ["metric_name", "condition"],
            as_index=False,
        )["score"]
        .mean()
        .pivot(
            index="metric_name",
            columns="condition",
            values="score",
        )
        .reset_index()
    )

# ============================================================
# Combined comparison
# ============================================================

if all_score_frames:
    all_scores = pd.concat(
        all_score_frames,
        ignore_index=True,
    )

    overall = (
        all_scores.groupby(
            ["metric_name", "condition"],
            as_index=False,
        )
        .agg(
            mean_score=("score", "mean"),
            median_score=("score", "median"),
            standard_deviation=("score", "std"),
            scored_outputs=("score", "count"),
        )
    )

    overall_wide = overall.pivot(
        index="metric_name",
        columns="condition",
        values="mean_score",
    ).reset_index()

    # Positive improvement always favours Full System.
    # TER is the only lower-is-better metric in this set.
    overall_wide["Full System improvement"] = (
        overall_wide["Baseline"]
        - overall_wide["Full System"]
    )

    higher_is_better = (
        overall_wide["metric_name"] != "ter"
    )

    overall_wide.loc[
        higher_is_better,
        "Full System improvement",
    ] = (
        overall_wide.loc[
            higher_is_better,
            "Full System",
        ]
        - overall_wide.loc[
            higher_is_better,
            "Baseline",
        ]
    )

    overall_wide["preferred_condition"] = (
        overall_wide["Full System improvement"]
        .apply(
            lambda value: (
                "Full System"
                if value > 0
                else "Baseline"
                if value < 0
                else "Tie"
            )
        )
    )

    by_dataset = (
        all_scores.groupby(
            ["dataset_id", "metric_name", "condition"],
            as_index=False,
        )["score"]
        .mean()
        .pivot(
            index=["dataset_id", "metric_name"],
            columns="condition",
            values="score",
        )
        .reset_index()
    )

    by_dataset["Full System improvement"] = (
        by_dataset["Baseline"]
        - by_dataset["Full System"]
    )

    higher_is_better = (
        by_dataset["metric_name"] != "ter"
    )

    by_dataset.loc[
        higher_is_better,
        "Full System improvement",
    ] = (
        by_dataset.loc[
            higher_is_better,
            "Full System",
        ]
        - by_dataset.loc[
            higher_is_better,
            "Baseline",
        ]
    )

    overall_wide.to_csv(
        overall_table_path,
        index=False,
    )
    by_dataset.to_csv(
        dataset_table_path,
        index=False,
    )

    print("\nOverall Full System vs Baseline")
    display(
        overall_wide.sort_values("metric_name")
    )

    print("\nPer-dataset Full System vs Baseline")
    display(
        by_dataset.sort_values(
            ["dataset_id", "metric_name"]
        )
    )

    print("\nPrimary dissertation metrics")
    primary_metrics = [
        "bertscore",
        "chrf",
        "meteor",
        "parent",
        "hhem",
        "alignscore",
    ]

    display(
        overall_wide[
            overall_wide["metric_name"].isin(primary_metrics)
        ].sort_values("metric_name")
    )

    print("\nSaved artifacts")
    print("Metrics input:", metrics_generations_path)
    print("Reference metrics:", reference_results_path)
    print("Source-grounded metrics:", source_results_path)
    print("Overall comparison:", overall_table_path)
    print("Per-dataset comparison:", dataset_table_path)

[03:32:38] Prepared 50 post-generation metric records.
[03:32:38] True references were added only to the metrics copy.
[03:32:38] Starting reference-alignment metrics...
[03:32:58] Still running reference-alignment metrics; elapsed=20.0s
[03:33:18] Still running reference-alignment metrics; elapsed=40.1s
[03:33:38] Still running reference-alignment metrics; elapsed=60.2s


/Users/realgobs/Documents/MScproject/table2text_pydanticai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RobertaModel were not initialized from the model checkpoint at /Users/realgobs/.cache/huggingface/hub/models--roberta-base/snapshots/e2da8e2f811d1448a5b465c236feacd80ffbac7b and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[03:33:58] Still running reference-alignment metrics; elapsed=80.3s
[03:34:19] Still running reference-alignment metrics; elapsed=100.4s
[03:34:39] Still running reference-alignment metrics; elapsed=120.5s
[03:34:49] Finished reference-alignment metrics after 131.0s

Reference-alignment metrics


condition,metric_name,Baseline,Full System
0,bertscore_f1,0.908855,0.923885
1,bleu,0.245494,0.312599
2,chrf,0.493101,0.551794
3,corpus_bleu,0.243208,0.309671
4,corpus_chrf,0.486392,0.540775
5,corpus_ter,1.978914,1.024585
6,meteor,0.419428,0.459503
7,rouge1,0.584656,0.664793
8,rouge2,0.369737,0.418757
9,rougeL,0.460632,0.496673


[03:34:49] Starting source-grounded HHEM and AlignScore...


...


Per-dataset Full System vs Baseline


condition,dataset_id,metric_name,Baseline,Full System,Full System improvement
0,dart,alignscore_base,0.690838,0.829053,0.138215
1,dart,bertscore_f1,0.947010,0.956011,0.009002
2,dart,bleu,0.353370,0.471387,0.118017
3,dart,chrf,0.652259,0.679864,0.027606
4,dart,corpus_bleu,0.365749,0.504011,0.138262
...,...,...,...,...,...
75,web_nlg,rouge1,0.779239,0.816837,0.037598
76,web_nlg,rouge2,0.612801,0.654265,0.041464
77,web_nlg,rougeL,0.736570,0.715242,-0.021328
78,web_nlg,rougeLsum,0.736570,0.715242,-0.021328



Primary dissertation metrics


condition,metric_name,Baseline,Full System,Full System improvement,preferred_condition
3,chrf,0.493101,0.551794,0.058693,Full System
10,meteor,0.419428,0.459503,0.040075,Full System



Saved artifacts
Metrics input: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/comparison/full_system_and_baseline_for_metrics.jsonl
Reference metrics: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/results/reference_alignment_metrics.jsonl
Source-grounded metrics: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/results/source_grounded_metrics.jsonl
Overall comparison: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/results/automatic_metrics_overall.csv
Per-dataset comparison: /Users/realgobs/Documents/MScproject/table2text_pydanticai/evaluation/protected_holdout_baseline/results/automatic_metrics_by_dataset.csv
